# V10: one model, up to 100 epochs, YOLO11m-seg, solar-disk normalization + crop refiner

V8 trains **one** YOLO11m-seg detector (up to 100 epochs; early stopping after 20 epochs without improvement) and **one**
ResNet34 U-Net crop refiner (up to 50 epochs; early stopping after 20 epochs without improvement). There are no folds and no ensemble.
With `TRAIN_ON_HOLDOUT = True` (the default) the model trains on **all labeled days
except about 8% reserved as tuning days** (652 of 707 photos train, 55 tune). The tuning
days select the YOLO and refiner checkpoints and the detector confidence and output
mode; there is no untouched holdout, so reported scores are tuning scores and are
optimistic. `TRAIN_ON_HOLDOUT = False` restores the 72% / 8% / 20% split and scores the
20% holdout once. The same model predicts the test images. Run on the Kaggle
**GPU T4 x2** accelerator, Internet on, data attached.

**No automatic upload.** The last cells write and check the CSV and print the submit command; you submit it yourself. The final check only prints warnings and never stops the run.

## New: solar-disk normalization (`DISK_NORMALIZE = True`)

Each H-alpha frame is a disk plus a dark or hazy surround, and the limb is a thin ring
far brighter than the disk (about 215 vs 120 gray levels). Before training, every
train and test frame is rewritten (geometry unchanged, still 2048 x 2048):

1. **Find the disk.** For 720 angles around the frame center, locate the brightest limb
   ring and the steepest drop just outside it; fit a circle with outlier trimming
   (center $c$, radius $R$). Fitted on all 887 frames: $R$ = 901-906 px, center within
   3 px of the frame center, every frame found.
2. **Remove the surround.** Pixels with $\lVert p-c\rVert > R+10$ are set to the neutral
   gray 128. The 10 px safety margin keeps all labels: of 8,199 annotations, none
   extends beyond $R+10$ (the farthest vertex is 0.5 px beyond $R$).
3. **Normalize the disk.** With $m(\rho)$ the median brightness at radius $\rho$ (1 px
   bins, so limb darkening and the limb ring are removed; within 30 px of the rim a
   per-5-degree-sector median follows the ring's non-circularity) and $\sigma$ the robust
   spread (1.4826 x MAD) of the residual inside $0.95R$:

$$z=\frac{I-m(\rho)}{\sigma},\qquad v=\operatorname{clip}(128+32z,\ 0,\ 255)$$

Everything downstream reads these normalized frames: the YOLO dataset, the refiner's
instance crops, calibration, holdout and test inference. Set `DISK_NORMALIZE = False`
(the run name changes automatically) for the matched baseline on identical splits,
seeds and settings. For a clean comparison run both with `TRAIN_ON_HOLDOUT = False` and
compare `outer_pq_selected`; with all data, compare `tuning_pq` (55 photos, optimistic).

## How the instance crop works (unchanged)

For a detected mask with bounding box $w \times h$:

$$s=\min\bigl(2048,\ \max(96,\ \lceil 1.8\,\max(w,h)\rceil)\bigr),\qquad
x_0=\operatorname{clip}\!\Bigl(\operatorname{round}\tfrac{x_{\min}+x_{\max}+1-s}{2},\,0,\,2048-s\Bigr)$$

The same formula gives $y_0$. The $s \times s$ window is resized to 256 x 256
($\lambda=256/s$; median window 173 px, so $\lambda\approx1.48$). The refiner is trained
with $\mathcal L=\tfrac12\mathrm{BCE}+\tfrac12(1-\mathrm{Dice})$ on ground-truth instance
crops. Output modes calibrated on the tuning days: `yolo` (detector mask), `refined`
($\sigma(z)\ge\tau$ in the window) and `gated` (`refined` limited to the detector mask
dilated by 12 px). An empty refinement falls back to the detector mask.

## Batch size and GPUs

YOLO trains with DDP on both T4s: batch 8 is 4 images per GPU. Ultralytics steps the
optimizer every `round(nbs/batch)` iterations: `nbs=32` gives a step every 4 iterations
(32 images, about 33 steps per epoch) and `warmup_epochs=1`.
The refiner trains with DataParallel on both GPUs (batch 32); set `refiner_gpus=[]` to
use one. Calibration, holdout and test inference run on GPU 0 and are limited by
CPU-side mask handling.


In [ ]:
%pip install -q 'ultralytics==8.4.211' 'segmentation-models-pytorch>=0.3.3,<0.7' pycocotools scipy 'opencv-python-headless>=4.9,<5' PyYAML

In [ ]:
import hashlib
import json
import random
import subprocess
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pycocotools.mask as mask_utils
import torch
from scipy.optimize import linear_sum_assignment
from PIL import Image

VERSION = "V10"
DISK_NORMALIZE = True  # False reproduces the raw-pixel baseline on the same split.
TRAIN_ON_HOLDOUT = True  # True: train on every labeled day except the tuning days.
PREPROCESSING = "disk_normalized" if DISK_NORMALIZE else "raw"
V8 = dict(
    weights="yolo11m-seg.pt", image_size=1280, batch_size=8, nbs=32, warmup_epochs=1,
    epochs=100, patience=20, learning_rate=0.002, workers=2,
    crop_size=256, crop_context=1.8, minimum_crop=96,
    refiner_encoder="resnet34", refiner_epochs=50, refiner_batch=32, refiner_gpus=[0, 1],
    refiner_lr=3e-4, refiner_patience=20, refiner_min_delta=1e-4,
    refiner_inference_batch=16,
    refiner_workers=0,  # crops are cached as small .npy files; avoids spawned loader processes
    tuning_images=64, confidences=[0.15, 0.25, 0.35],
    refiner_thresholds=[0.5, 0.65], gate_pixels=[6, 12], min_areas=[64],
    disk_margin=10, preprocess_workers=4,
)
CFG = dict(seed=2026)
SPLIT_FOLD = 0  # one of the five day-grouped splits; only this one is trained.
DATA_TAG = "alldata" if TRAIN_ON_HOLDOUT else "holdout"
SETTINGS_TAG = hashlib.sha1(json.dumps(V8, sort_keys=True).encode()).hexdigest()[:6]  # any edit gives a new run folder
RUN_NAME = (f"v10_single_{PREPROCESSING}_{DATA_TAG}_{Path(V8['weights']).stem}_"
            f"{V8['epochs']}ep_img{V8['image_size']}_{SETTINGS_TAG}")
SMOKE_TEST = False
DATA_ROOT_OVERRIDE = None
OUTPUT_DIR = (Path("/kaggle/working") if Path("/kaggle/working").is_dir()
              else Path.cwd() / "cv_outputs") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CV_ROOT = OUTPUT_DIR
# Helper code, normalized frames and the crop cache are scratch: keep them out of the Output tab.
SCRATCH_DIR = (Path("/kaggle/temp") / RUN_NAME if Path("/kaggle/temp").is_dir()
               else CV_ROOT / "_scratch")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
CROP_CACHE_DIR = SCRATCH_DIR / "_crop_cache"
COMPETITION = "filament-segmentation-2026"
DATASET_NAME = "MAGFiLO_1.0_Kaggle_2026"
ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
FULL_SHAPE = (2048, 2048)
FOLDS_TO_RUN = [SPLIT_FOLD]
INNER_TUNE_FRACTION = 0.1
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    YOLO_TRAIN_DEVICE = [0, 1]
    YOLO_INFERENCE_DEVICE = 0
    REFINER_DEVICE = torch.device("cuda:0")
elif SMOKE_TEST:
    YOLO_TRAIN_DEVICE = YOLO_INFERENCE_DEVICE = "cpu"
    REFINER_DEVICE = torch.device("cpu")
else:
    raise RuntimeError("V8 requires two visible GPUs (Kaggle T4x2) for batch size 8.")

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])
print("Preprocessing:", PREPROCESSING, "| training devices:", YOLO_TRAIN_DEVICE,
      "| inference device:", YOLO_INFERENCE_DEVICE, "| refiner device:", REFINER_DEVICE,
      "| outputs:", CV_ROOT, "| scratch:", SCRATCH_DIR)


In [ ]:
import os
import shutil

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("CPU cores:", os.cpu_count(),
      "| free disk in output dir: %.1f GB (this run writes about 3 GB)"
      % (shutil.disk_usage(OUTPUT_DIR).free / 1e9))
if shutil.disk_usage(OUTPUT_DIR).free < 15e9:
    print("WARNING: under 15 GB free. Delete old run folders first, for example: "
          "!rm -rf /kaggle/working/v*_single_*")
try:
    import segmentation_models_pytorch as smp
    import ultralytics
    print("ultralytics", ultralytics.__version__, "| segmentation_models_pytorch", smp.__version__)
except Exception as error:  # report only; never stop Run All
    print("WARNING: a required package did not import:", error)


In [ ]:
candidates = ([Path(DATA_ROOT_OVERRIDE)] if DATA_ROOT_OVERRIDE else [
    Path('/kaggle/input/competitions') / COMPETITION / DATASET_NAME,
    Path('/kaggle/input') / COMPETITION / DATASET_NAME,
    Path.cwd() / 'data' / DATASET_NAME, Path.cwd() / DATASET_NAME])
if DATA_ROOT_OVERRIDE is None and Path('/kaggle/input').is_dir():
    candidates += sorted(Path('/kaggle/input').glob(f'*/{DATASET_NAME}'))
    candidates += sorted(Path('/kaggle/input').glob(f'*/*/{DATASET_NAME}'))
valid_roots = [p for p in candidates if (p / 'train' / ANNOTATION_NAME).is_file()
               and (p / 'test' / 'test_images').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Attach the competition data or set DATA_ROOT_OVERRIDE to its dataset root.')
DATA_ROOT = valid_roots[0]
train_paths = sorted((DATA_ROOT / 'train' / 'train_images').glob('*.jpeg'))
test_paths = sorted((DATA_ROOT / 'test' / 'test_images').glob('*.jpeg'))
if not train_paths or not test_paths:
    raise FileNotFoundError('Train and test must both contain JPEG files.')
with (DATA_ROOT / 'train' / ANNOTATION_NAME).open() as stream:
    coco = json.load(stream)
records = coco['images']
if len({r['id'] for r in records}) != len(records):
    raise ValueError('Duplicate COCO image IDs.')
records_by_name = defaultdict(list)
annotations_by_id = defaultdict(list)
for record in records:
    if (record['height'], record['width']) != FULL_SHAPE:
        raise ValueError(f'Unexpected dimensions for {record["id"]}')
    records_by_name[record['file_name']].append(record)
for ann in coco['annotations']:
    if ann.get('iscrowd', 0) != 0:
        raise ValueError('Local PQ assumes individual non-crowd filaments.')
    annotations_by_id[ann['image_id']].append(ann)
known_ids = {r['id'] for r in records}
if set(annotations_by_id) - known_ids:
    raise ValueError('Orphan annotation image IDs.')
if len({a['id'] for a in coco['annotations']}) != len(coco['annotations']):
    raise ValueError('Duplicate filament annotation IDs.')
train_lookup = {p.name: p for p in train_paths}
if set(records_by_name) - set(train_lookup):
    raise FileNotFoundError('Annotations reference missing training files.')
if set(train_lookup) & {p.name for p in test_paths}:
    raise ValueError('Train/test filenames overlap.')

annotation_sha256 = hashlib.sha256((DATA_ROOT / 'train' / ANNOTATION_NAME).read_bytes()).hexdigest()
print('Dataset:', len(train_paths), 'labeled physical images;', len(test_paths), 'unlabeled test images')


In [ ]:
def ann_rle(annotation):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        if not segmentation or any(len(p) < 6 or len(p) % 2 for p in segmentation):
            raise ValueError(f"Invalid polygon: {annotation['id']}")
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, *FULL_SHAPE))
    elif isinstance(segmentation, dict):
        if list(segmentation["size"]) != list(FULL_SHAPE):
            raise ValueError("RLE dimensions do not match the image.")
        if isinstance(segmentation["counts"], list):
            rle = mask_utils.frPyObjects(segmentation, *FULL_SHAPE)
        else:
            counts = segmentation["counts"]
            rle = dict(size=list(FULL_SHAPE), counts=counts.encode("ascii") if isinstance(counts, str) else counts)
    else:
        raise TypeError(f"Unsupported segmentation: {annotation['id']}")
    if mask_utils.area(rle) <= 0:
        raise ValueError(f"Empty ground-truth filament: {annotation['id']}")
    return rle


def gt_rles(image_id):
    return [ann_rle(annotation) for annotation in annotations_by_id[image_id]]


def local_scores(predicted, truth):
    n_pred, n_true = len(predicted), len(truth)
    overlaps = (mask_utils.iou(predicted, truth, [0] * n_true)
                if n_pred and n_true else np.zeros((n_pred, n_true)))
    matched = []
    if n_pred and n_true:
        valid = overlaps > 0.5
        rewards = valid * (1 + overlaps / (min(n_pred, n_true) + 1))
        rows, columns = linear_sum_assignment(-rewards)
        matched = [float(overlaps[r, c]) for r, c in zip(rows, columns) if valid[r, c]]
    tp = len(matched)
    fp, fn = n_pred - tp, n_true - tp
    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = sum(matched) / denominator if denominator else 1.0
    pred_union = mask_utils.decode(mask_utils.merge(predicted)) if predicted else np.zeros(FULL_SHAPE, dtype=np.uint8)
    true_union = mask_utils.decode(mask_utils.merge(truth)) if truth else np.zeros(FULL_SHAPE, dtype=np.uint8)
    intersection = np.count_nonzero(pred_union.astype(bool) & true_union.astype(bool))
    union = np.count_nonzero(pred_union.astype(bool) | true_union.astype(bool))
    pixels = np.count_nonzero(pred_union) + np.count_nonzero(true_union)
    return dict(pq=pq, dice=2 * intersection / pixels if pixels else 1.0,
                iou=intersection / union if union else 1.0, tp=tp, fp=fp, fn=fn)


In [ ]:
import sys
from pathlib import Path

runtime_dir = SCRATCH_DIR / "_runtime"
runtime_dir.mkdir(parents=True, exist_ok=True)
(runtime_dir / "grouped_yolo_cv.py").write_text('"""Day-grouped outer CV with inner tuning for the standalone YOLO/refiner notebook."""\n\nimport hashlib\nimport csv\nimport json\nimport math\nimport random\nfrom collections import defaultdict\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\nfrom scipy.optimize import linear_sum_assignment\n\n\ndef cv_digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef make_day_folds(names, seed=2026, folds=5, inner_fraction=0.1):\n    names = sorted(names)\n    if len(set(names)) != len(names) or folds != 5 or not 0 < inner_fraction < 1:\n        raise ValueError("Use unique physical filenames, five folds, and 0 < inner_fraction < 1.")\n    by_day = defaultdict(list)\n    for name in names:\n        datetime.strptime(name[:8], "%Y%m%d")\n        by_day[name[:8]].append(name)\n    if len(by_day) < 10:\n        raise ValueError("Five-fold CV needs at least ten observation days.")\n    days = sorted(by_day)\n    random.Random(seed).shuffle(days)\n    # Largest physical-image groups first; seeded order breaks equal-size ties.\n    days.sort(key=lambda day: -len(by_day[day]))\n    outer_days, sizes = [[] for _ in range(folds)], [0] * folds\n    for day in days:\n        fold = min(range(folds), key=lambda index: (sizes[index], index))\n        outer_days[fold].append(day)\n        sizes[fold] += len(by_day[day])\n    plans = []\n    for fold in range(folds):\n        remaining = sorted(set(days) - set(outer_days[fold]))\n        random.Random(seed + fold + 1).shuffle(remaining)\n        n_inner = max(1, round(len(remaining) * inner_fraction))\n        if n_inner >= len(remaining):\n            raise ValueError("Inner tuning leaves no training days.")\n        groups = dict(\n            train=set(remaining[n_inner:]), tune=set(remaining[:n_inner]),\n            holdout=set(outer_days[fold]),\n        )\n        split = {key: sorted(name for day in selected for name in by_day[day])\n                 for key, selected in groups.items()}\n        if any(not selected for selected in split.values()):\n            raise ValueError("Every CV partition must be nonempty.")\n        if set.union(*groups.values()) != set(by_day):\n            raise ValueError("CV partitions do not cover the labeled dataset.")\n        if any(groups[a] & groups[b] for a, b in\n               [("train", "tune"), ("train", "holdout"), ("tune", "holdout")]):\n            raise ValueError("Observation-day leakage.")\n        plans.append(dict(fold=fold, splits=split))\n    outer = [name for plan in plans for name in plan["splits"]["holdout"]]\n    if len(outer) != len(names) or set(outer) != set(names):\n        raise ValueError("Each physical image must appear in exactly one outer fold.")\n    return plans\n\n\ndef save_cv_plan(path, plan):\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != plan:\n            raise ValueError("Existing CV plan differs. Choose a new CV_RUN_NAME.")\n    else:\n        path.write_text(json.dumps(plan, indent=2, allow_nan=False))\n\n\ndef fold_records(plan, records_by_name):\n    return {phase: [record for name in names for record in records_by_name[name]]\n            for phase, names in plan["splits"].items()}\n\n\ndef validate_fold_result(result, plan, plan_hash):\n    if (result["plan_sha256"] != plan_hash or result["fold"] != plan["fold"]\n            or result["status"] != "complete"):\n        raise ValueError("Fold result provenance/status mismatch.")\n    rows = result["outer_rows"]\n    expected = plan["splits"]["holdout"]\n    actual = [row["filename"] for row in rows]\n    if len(actual) != len(set(actual)) or set(actual) != set(expected):\n        raise ValueError("Outer rows must cover exactly this fold\'s physical images.")\n    if not result["checkpoint_hashes"] or not result["selected_parameters"]:\n        raise ValueError("Missing selected model provenance.")\n    for row in rows:\n        if row["annotator_sets"] < 1:\n            raise ValueError("Missing outer annotator evaluation.")\n        for key in ["pq", "dice", "iou"]:\n            if not isinstance(row[key], (int, float)) or not math.isfinite(row[key]) or not 0 <= row[key] <= 1:\n                raise ValueError("Invalid outer metric.")\n    mean = float(np.mean([row["pq"] for row in rows]))\n    if not math.isclose(mean, result["outer_mean"]["pq"], abs_tol=1e-12):\n        raise ValueError("Outer PQ summary disagrees with per-image rows.")\n\n\ndef summarize_folds(results, plans, plan_hash):\n    by_fold = {}\n    for result in results:\n        fold = result["fold"]\n        if fold not in range(5) or fold in by_fold:\n            raise ValueError("Unknown or duplicate completed fold.")\n        validate_fold_result(result, plans[fold], plan_hash)\n        by_fold[fold] = result\n    complete = len(by_fold) == 5\n    summary = dict(\n        plan_sha256=plan_hash, complete=complete, completed_folds=sorted(by_fold),\n        missing_folds=sorted(set(range(5)) - set(by_fold)),\n        status="complete" if complete else "incomplete: no five-fold aggregate reported",\n    )\n    if complete:\n        fold_pq = [by_fold[fold]["outer_mean"]["pq"] for fold in range(5)]\n        rows = [row for fold in range(5) for row in by_fold[fold]["outer_rows"]]\n        summary.update(\n            fold_mean_pq=float(np.mean(fold_pq)),\n            fold_sample_sd_pq=float(np.std(fold_pq, ddof=1)),\n            pooled_oof_mean_pq=float(np.mean([row["pq"] for row in rows])),\n            physical_images=len(rows),\n            note="Per-image metrics average annotators first. Fold SD is descriptive, not a confidence interval. "\n                 "These are reused labeled data, not a new untouched final test or leaderboard score.",\n        )\n    return summary\n\n\ndef collect_fold_results(roots, plans, plan_hash):\n    results = {}\n    for root in roots:\n        for path in sorted(Path(root).rglob("fold_result.json")):\n            result = json.loads(path.read_text())\n            fold = result["fold"]\n            if fold not in range(5):\n                raise ValueError(f"Unknown fold in {path}")\n            validate_fold_result(result, plans[fold], plan_hash)\n            if fold in results and results[fold] != result:\n                raise ValueError(f"Conflicting results for fold {fold}; do not select by outer score.")\n            results[fold] = result\n    return [results[fold] for fold in sorted(results)]\n\n\ndef evaluate_outer(names, parameters, namespace, detector, refiner):\n    rows = []\n    for name in namespace["tqdm"](names, desc="Outer evaluation: frozen parameters"):\n        proposals = namespace["v4_proposals"](\n            namespace["train_lookup"][name], detector, refiner, parameters["confidence"],\n        )\n        predicted, fallbacks = namespace["v4_instances"](proposals, parameters)\n        namespace["validate_disjoint"](predicted, namespace["FULL_SHAPE"])\n        scores = [namespace["local_scores"](predicted, namespace["gt_rles"](record["id"]))\n                  for record in namespace["records_by_name"][name]]\n        if not scores:\n            raise ValueError("No annotator set for outer image.")\n        rows.append(dict(\n            filename=name, annotator_sets=len(scores), instances=len(predicted),\n            refiner_fallbacks=fallbacks,\n            **{key: float(np.mean([score[key] for score in scores])) for key in scores[0]},\n        ))\n    if not rows:\n        raise ValueError("No outer images.")\n    means = {key: float(np.mean([row[key] for row in rows]))\n             for key in rows[0] if key not in ["filename", "annotator_sets"]}\n    return means, rows\n\n\ndef run_cv_fold(namespace, plan, shared_plan, root):\n    """Run one isolated fold; outer labels never enter training or calibration."""\n    plan_hash = cv_digest(shared_plan)\n    directory = Path(root) / f"fold_{plan[\'fold\']}"\n    directory.mkdir(parents=True, exist_ok=True)\n    result_path = directory / "fold_result.json"\n    if result_path.exists():\n        result = json.loads(result_path.read_text())\n        validate_fold_result(result, plan, plan_hash)\n        print("Reusing completed outer evaluation:", result_path)\n        return result\n    namespace["OUTPUT_DIR"] = directory\n    namespace["split_names"] = plan["splits"]\n    namespace["split_records"] = fold_records(plan, namespace["records_by_name"])\n    namespace["manifest"] = dict(\n        seed=namespace["CFG"]["seed"], grouping="observation-day",\n        annotation_sha256=namespace["annotation_sha256"], splits=plan["splits"],\n    )\n    namespace["write_json"](directory / "split_manifest.json", namespace["manifest"])\n    for flag, files in [\n        ("RUN_YOLO_TRAINING", ["yolo_best.pt", "yolo_provenance.json"]),\n        ("RUN_REFINER_TRAINING", ["refiner_best.pt", "refiner_runtime.json"]),\n    ]:\n        existing = [(directory / name).exists() for name in files]\n        if any(existing) and not all(existing):\n            raise ValueError(f"Incomplete checkpoint evidence in {directory}; recover it explicitly.")\n        namespace[flag] = not all(existing)\n    namespace["RUN_V4_CALIBRATION"] = not (directory / "v4_calibration.json").exists()\n    detector = refiner = None\n    try:\n        detector = namespace["v4_train_yolo"]()\n        refiner = namespace["v4_train_refiner"]()\n        calibration = namespace["v4_calibrate"](detector, refiner)\n        means, rows = evaluate_outer(\n            plan["splits"]["holdout"], calibration["selected"]["parameters"],\n            namespace, detector, refiner,\n        )\n        result = dict(\n            fold=plan["fold"], plan_sha256=plan_hash, status="complete",\n            smoke_test=shared_plan["smoke_test"],\n            checkpoint_hashes=calibration["hashes"],\n            selected_parameters=calibration["selected"]["parameters"],\n            inner_selected_scores=calibration["selected"]["scores"],\n            outer_mean=means, outer_rows=rows,\n        )\n        validate_fold_result(result, plan, plan_hash)\n        namespace["write_json"](directory / "fold_result.json", result)\n        return result\n    finally:\n        del detector, refiner\n        namespace["gc"].collect()\n        if namespace["torch"].cuda.is_available():\n            namespace["torch"].cuda.empty_cache()\n\n\ndef export_cv_submission(namespace, plan, shared_plan, root):\n    """Export a predeclared fold; never select a model by outer score."""\n    directory = Path(root) / f"fold_{plan[\'fold\']}"\n    result = json.loads((directory / "fold_result.json").read_text())\n    validate_fold_result(result, plan, cv_digest(shared_plan))\n    namespace.update(\n        OUTPUT_DIR=directory, split_names=plan["splits"],\n        split_records=fold_records(plan, namespace["records_by_name"]),\n        RUN_YOLO_TRAINING=False, RUN_REFINER_TRAINING=False, RUN_V4_CALIBRATION=False,\n    )\n    detector = refiner = None\n    try:\n        detector = namespace["v4_train_yolo"]()\n        refiner = namespace["v4_train_refiner"]()\n        calibration = namespace["v4_calibrate"](detector, refiner)\n        if calibration["hashes"] != result["checkpoint_hashes"]:\n            raise ValueError("Submission checkpoints differ from the completed fold.")\n        return namespace["v4_export"](detector, refiner, calibration)\n    finally:\n        del detector, refiner\n        namespace["gc"].collect()\n        if namespace["torch"].cuda.is_available():\n            namespace["torch"].cuda.empty_cache()\n\n\nENSEMBLE_POLICY = dict(\n    method="equal-fold-instance-majority-v1", match_iou=0.5,\n    required_fold_votes=3, pixel_votes=3, min_area=20,\n)\n\n\ndef fuse_fold_instances(fold_instances, shape):\n    """Match at most one instance per fold per cluster, then vote across five folds."""\n    if len(fold_instances) != 5:\n        raise ValueError("Ensembling requires predictions from exactly five folds.")\n    clusters = []\n    for instances in fold_instances:\n        for rle in instances:\n            if tuple(rle["size"]) != tuple(shape) or not mask_utils.area(rle):\n                raise ValueError("Invalid ensemble instance shape or empty mask.")\n        if not instances:\n            continue\n        representatives = []\n        for cluster in clusters:\n            votes = sum((mask_utils.decode(rle).astype(np.uint8) for rle in cluster),\n                        np.zeros(shape, dtype=np.uint8))\n            representatives.append(mask_utils.encode(np.asfortranarray(\n                (votes * 2 >= len(cluster)).astype(np.uint8))))\n        matched = set()\n        if representatives:\n            overlaps = mask_utils.iou(representatives, instances, [0] * len(instances))\n            eligible = overlaps >= ENSEMBLE_POLICY["match_iou"]\n            # Maximize valid match count first, then IoU; invalid pairs cannot steal matches.\n            weights = eligible * (min(overlaps.shape) + 1 + overlaps)\n            rows, columns = linear_sum_assignment(-weights)\n            for row, column in zip(rows, columns):\n                if eligible[row, column]:\n                    clusters[row].append(instances[column])\n                    matched.add(column)\n        clusters.extend([rle] for index, rle in enumerate(instances) if index not in matched)\n    fused = []\n    for cluster in clusters:\n        if len(cluster) < ENSEMBLE_POLICY["required_fold_votes"]:\n            continue\n        votes = sum((mask_utils.decode(rle).astype(np.uint8) for rle in cluster),\n                    np.zeros(shape, dtype=np.uint8))\n        mask = (votes >= ENSEMBLE_POLICY["pixel_votes"]).astype(np.uint8)\n        if int(mask.sum()) >= ENSEMBLE_POLICY["min_area"]:\n            fused.append((len(cluster), int(mask.sum()),\n                          mask_utils.encode(np.asfortranarray(mask))))\n    fused.sort(key=lambda item: (-item[0], -item[1]))\n    return [rle for _, _, rle in fused]\n\n\ndef export_cv_ensemble(namespace, plans, shared_plan, root):\n    """Reload all five models sequentially and export one validated voted submission."""\n    if shared_plan["smoke_test"] or namespace["SMOKE_TEST"]:\n        raise ValueError("Smoke runs cannot produce competition exports.")\n    if sorted(plan["fold"] for plan in plans) != list(range(5)):\n        raise ValueError("All five unique fold plans are required.")\n    plans = sorted(plans, key=lambda plan: plan["fold"])\n    root = Path(root)\n    results = []\n    for plan in plans:\n        result = json.loads((root / f"fold_{plan[\'fold\']}" / "fold_result.json").read_text())\n        validate_fold_result(result, plan, cv_digest(shared_plan))\n        if result.get("smoke_test", False):\n            raise ValueError("Cannot ensemble smoke checkpoints.")\n        results.append(result)\n    test_paths = namespace["test_paths"]\n    if not test_paths or len({path.stem for path in test_paths}) != len(test_paths):\n        raise ValueError("Test image IDs must be nonempty and unique.")\n    prediction_paths = []\n    for plan, result in zip(plans, results):\n        directory = root / f"fold_{plan[\'fold\']}"\n        namespace.update(\n            OUTPUT_DIR=directory, split_names=plan["splits"],\n            split_records=fold_records(plan, namespace["records_by_name"]),\n            RUN_YOLO_TRAINING=False, RUN_REFINER_TRAINING=False, RUN_V4_CALIBRATION=False,\n        )\n        detector = refiner = None\n        try:\n            detector = namespace["v4_train_yolo"]()\n            refiner = namespace["v4_train_refiner"]()\n            calibration = namespace["v4_calibrate"](detector, refiner)\n            if (calibration["hashes"] != result["checkpoint_hashes"]\n                    or calibration["selected"]["parameters"] != result["selected_parameters"]):\n                raise ValueError("Ensemble checkpoints/calibration differ from completed fold.")\n            parameters = calibration["selected"]["parameters"]\n            predictions = {}\n            for path in namespace["tqdm"](test_paths, desc=f"Ensemble fold {plan[\'fold\']} test"):\n                proposals = namespace["v4_proposals"](\n                    path, detector, refiner, parameters["confidence"])\n                instances, _ = namespace["v4_instances"](proposals, parameters)\n                namespace["validate_disjoint"](instances, namespace["FULL_SHAPE"])\n                predictions[path.stem] = [\n                    dict(size=list(rle["size"]), counts=rle["counts"].decode("ascii"))\n                    for rle in instances\n                ]\n            prediction_path = directory / "ensemble_predictions.json"\n            namespace["write_json"](prediction_path, predictions)\n            prediction_paths.append(prediction_path)\n        finally:\n            del detector, refiner\n            namespace["gc"].collect()\n            if namespace["torch"].cuda.is_available():\n                namespace["torch"].cuda.empty_cache()\n    predictions = [json.loads(path.read_text()) for path in prediction_paths]\n    rows, image_audits = [], []\n    for path in test_paths:\n        per_fold = [\n            [dict(size=rle["size"], counts=rle["counts"].encode("ascii"))\n             for rle in fold[path.stem]] for fold in predictions\n        ]\n        instances = fuse_fold_instances(per_fold, namespace["FULL_SHAPE"])\n        instances, ownership = namespace["disjoint_instances"](\n            instances, namespace["FULL_SHAPE"], ENSEMBLE_POLICY["min_area"])\n        namespace["validate_disjoint"](instances, namespace["FULL_SHAPE"])\n        for rle in instances:\n            if mask_utils.encode(np.asfortranarray(mask_utils.decode(rle)))["counts"] != rle["counts"]:\n                raise ValueError("Ensemble RLE roundtrip failed.")\n        image_audits.append(dict(\n            filename=path.name, fold_instances=[len(items) for items in per_fold],\n            instances=len(instances), ownership=ownership,\n        ))\n        rows.extend(dict(filament_id=f"{path.stem}_{index}",\n                         segmentation_rle=rle["counts"].decode("ascii"))\n                    for index, rle in enumerate(instances, 1))\n    audit = namespace["validate_submission_rows"](\n        rows, {path.stem for path in test_paths}, namespace["FULL_SHAPE"])\n    destination = root / "submission.csv"\n    temporary = root / "submission.ensemble.tmp.csv"\n    fields = ["filament_id", "segmentation_rle"]\n    try:\n        with temporary.open("w", newline="") as stream:\n            writer = csv.DictWriter(stream, fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n        with temporary.open(newline="") as stream:\n            reader = csv.DictReader(stream)\n            if reader.fieldnames != fields or list(reader) != rows:\n                raise ValueError("Ensemble CSV readback mismatch.")\n        namespace["write_json"](root / "ensemble_audit.json", dict(\n            policy=ENSEMBLE_POLICY, plan_sha256=cv_digest(shared_plan),\n            folds=[dict(fold=r["fold"], hashes=r["checkpoint_hashes"],\n                        parameters=r["selected_parameters"]) for r in results],\n            validation=audit, images=image_audits,\n            submission_sha256=hashlib.sha256(temporary.read_bytes()).hexdigest(),\n        ))\n        temporary.replace(destination)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    kaggle_root = Path("/kaggle/working")\n    if kaggle_root.is_dir() and destination.resolve() != (kaggle_root / "submission.csv").resolve():\n        namespace["shutil"].copy2(destination, kaggle_root / "submission.csv")\n    namespace["OUTPUT_DIR"] = root\n    if audit["zero_detection_images"]:\n        print("WARNING: zero-detection images omitted, no invented masks:",\n              audit["zero_detection_images"])\n    return destination\n')
(runtime_dir / "submission_masks.py").write_text('"""Disjoint instance masks and an offline repair command for saved V3 CSVs."""\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\n\n\nMASK_POLICY = "first-mask-wins-v1"\nSUBMISSION_FIELDS = ["filament_id", "segmentation_rle"]\n\n\ndef decode_instance(rle, shape):\n    if tuple(rle["size"]) != tuple(shape):\n        raise ValueError(f"Instance shape {rle[\'size\']} does not match {shape}.")\n    mask = mask_utils.decode(rle)\n    if mask.shape != tuple(shape) or not np.isin(mask, [0, 1]).all() or not mask.any():\n        raise ValueError("An instance must be a nonempty binary mask.")\n    return mask.astype(bool)\n\n\ndef disjoint_instances(instances, shape=(2048, 2048), min_area=1):\n    """Keep earlier masks unchanged; later masks may claim only unowned pixels."""\n    if min_area < 1:\n        raise ValueError("min_area must be positive.")\n    occupied = np.zeros(shape, dtype=bool)\n    result, kept_indices, dropped_indices = [], [], []\n    overlap_pixels = 0\n    for index, rle in enumerate(instances):\n        mask = decode_instance(rle, shape)\n        overlap_pixels += int(np.count_nonzero(mask & occupied))\n        mask &= ~occupied\n        if np.count_nonzero(mask) < min_area:\n            dropped_indices.append(index)\n            continue\n        occupied |= mask\n        result.append(mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))))\n        kept_indices.append(index)\n    return result, dict(\n        policy=MASK_POLICY, overlap_pixels_removed=overlap_pixels,\n        kept_indices=kept_indices, dropped_indices=dropped_indices,\n    )\n\n\ndef validate_disjoint(instances, shape=(2048, 2048)):\n    occupied = np.zeros(shape, dtype=bool)\n    for index, rle in enumerate(instances):\n        mask = decode_instance(rle, shape)\n        if np.any(occupied & mask):\n            raise ValueError(f"Instance {index} overlaps an earlier instance.")\n        occupied |= mask\n\n\ndef submission_groups(rows, test_stems, shape):\n    groups, seen = defaultdict(list), set()\n    for row in rows:\n        if set(row) != set(SUBMISSION_FIELDS):\n            raise ValueError("CSV must have exactly the two competition columns.")\n        identity = row["filament_id"]\n        stem, separator, number = identity.rpartition("_")\n        if (not separator or not number.isdigit() or int(number) < 1\n                or stem not in test_stems or identity in seen):\n            raise ValueError(f"Unknown, invalid, or duplicate filament_id: {identity}")\n        seen.add(identity)\n        rle = dict(size=list(shape), counts=row["segmentation_rle"].encode("ascii"))\n        groups[stem].append((row, rle))\n    if not groups:\n        raise ValueError("Submission has no predicted instances.")\n    return groups\n\n\ndef validate_submission_rows(rows, test_stems, shape=(2048, 2048)):\n    groups = submission_groups(rows, test_stems, shape)\n    for stem, entries in groups.items():\n        try:\n            validate_disjoint([rle for _, rle in entries], shape)\n        except ValueError as error:\n            raise ValueError(f"{stem}: {error}") from error\n    return dict(\n        rows=len(rows), images_with_predictions=len(groups),\n        zero_detection_images=sorted(set(test_stems) - set(groups)),\n        overlapping_pixels=0,\n    )\n\n\ndef repair_submission(source, destination, test_stems, min_area=20, shape=(2048, 2048)):\n    source, destination = Path(source), Path(destination)\n    audit_path = destination.with_suffix(".audit.json")\n    if source.resolve() == destination.resolve():\n        raise ValueError("Repair must not overwrite the original submission.")\n    if destination.exists() or audit_path.exists():\n        raise FileExistsError("Choose a new repair output path; existing evidence is preserved.")\n    with source.open(newline="") as stream:\n        reader = csv.DictReader(stream)\n        if reader.fieldnames != SUBMISSION_FIELDS:\n            raise ValueError("Unexpected submission columns or column order.")\n        original_rows = list(reader)\n    groups = submission_groups(original_rows, test_stems, shape)\n    repaired_by_id, image_audits = {}, []\n    for stem, entries in groups.items():\n        masks, audit = disjoint_instances([rle for _, rle in entries], shape, min_area)\n        for index, rle in zip(audit["kept_indices"], masks):\n            identity = entries[index][0]["filament_id"]\n            repaired_by_id[identity] = dict(\n                filament_id=identity, segmentation_rle=rle["counts"].decode("ascii"),\n            )\n        image_audits.append(dict(\n            image=stem, overlap_pixels_removed=audit["overlap_pixels_removed"],\n            input_instances=len(entries), output_instances=len(masks),\n            dropped_ids=[entries[i][0]["filament_id"] for i in audit["dropped_indices"]],\n        ))\n    rows = [repaired_by_id[r["filament_id"]] for r in original_rows\n            if r["filament_id"] in repaired_by_id]\n    validation = validate_submission_rows(rows, test_stems, shape)\n    temporary = destination.with_suffix(".tmp.csv")\n    if temporary.exists():\n        raise FileExistsError(f"Temporary file already exists: {temporary}")\n    try:\n        with temporary.open("x", newline="") as stream:\n            writer = csv.DictWriter(stream, fieldnames=SUBMISSION_FIELDS)\n            writer.writeheader()\n            writer.writerows(rows)\n        with temporary.open(newline="") as stream:\n            reader = csv.DictReader(stream)\n            if reader.fieldnames != SUBMISSION_FIELDS or list(reader) != rows:\n                raise ValueError("Repaired CSV readback mismatch.")\n        report = dict(\n            policy=MASK_POLICY, min_area=min_area, validation=validation,\n            input_csv=str(source), input_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),\n            output_sha256=hashlib.sha256(temporary.read_bytes()).hexdigest(),\n            input_rows=len(original_rows), dropped_rows=len(original_rows) - len(rows),\n            images_with_overlap=sum(a["overlap_pixels_removed"] > 0 for a in image_audits),\n            overlap_pixels_removed=sum(a["overlap_pixels_removed"] for a in image_audits),\n            images=image_audits,\n            note="Offline repair only. No retraining, recalibration, or submission. "\n                 "Previous tuning scores do not measure these repaired masks.",\n        )\n        with audit_path.open("x") as stream:\n            stream.write(json.dumps(report, indent=2, allow_nan=False))\n        temporary.replace(destination)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("source", type=Path)\n    parser.add_argument("destination", type=Path)\n    parser.add_argument("--test-dir", type=Path, required=True)\n    parser.add_argument("--min-area", type=int, default=20)\n    args = parser.parse_args()\n    test_stems = {p.stem for p in args.test_dir.rglob("*.jpeg")}\n    if not test_stems:\n        parser.error("--test-dir contains no test JPEGs.")\n    report = repair_submission(args.source, args.destination, test_stems, args.min_area)\n    print(json.dumps({key: value for key, value in report.items() if key != "images"}, indent=2))\n    print("REPAIRED CSV:", args.destination, "No competition upload performed.")\n\n\nif __name__ == "__main__":\n    main()\n')
(runtime_dir / "solar_disk.py").write_text('"""Solar-disk detection and per-image normalization for H-alpha frames."""\n\nimport cv2\nimport numpy as np\nfrom scipy import ndimage as ndi\n\nBIN_PIXELS = 1\nOUTPUT_CENTER = 128\nOUTPUT_SCALE = 32.0\nRIM_ZONE = 30\nSECTORS = 72\nMIN_BIN_SAMPLES = 20\nSMOOTH_SIGMA = 4.0\nTOPHAT_KERNEL = 31\nTOPHAT_SIGMA = 1.5\nRIDGE_GAIN = 1.4\n\n\ndef _fit_circle(points):\n    x, y = points[:, 0], points[:, 1]\n    design = np.column_stack([2 * x, 2 * y, np.ones_like(x)])\n    solution, *_ = np.linalg.lstsq(design, x ** 2 + y ** 2, rcond=None)\n    cx, cy = solution[0], solution[1]\n    return cx, cy, float(np.sqrt(max(solution[2] + cx ** 2 + cy ** 2, 0.0)))\n\n\ndef _limb_points(gray, cx, cy, radii, angles):\n    cos, sin = np.cos(angles)[:, None], np.sin(angles)[:, None]\n    samples = ndi.map_coordinates(\n        gray, [cy + radii[None, :] * sin, cx + radii[None, :] * cos], order=1, mode="nearest")\n    samples = ndi.gaussian_filter1d(samples, 1.5, axis=1)\n    # The limb ring is the brightest feature; take the steepest drop just outside it.\n    position = np.arange(samples.shape[1])[None, :]\n    peak = np.argmax(samples, axis=1)[:, None]\n    gradient = np.where((position >= peak) & (position <= peak + 20),\n                        np.gradient(samples, axis=1), np.inf)\n    edge = radii[np.argmin(gradient, axis=1)]\n    return np.column_stack([cx + edge * np.cos(angles), cy + edge * np.sin(angles)])\n\n\ndef fit_disk(gray, passes=2):\n    """Return (cx, cy, radius, ok): the circle at the steepest outward brightness drop.\n\n    The limb is a thin bright ring. A threshold or the global steepest drop can lock\n    onto a bright halo\'s outer edge in some frames, so the rim is taken from the\n    ring itself.\n    """\n    gray = np.asarray(gray, dtype=np.float32)\n    height, width = gray.shape\n    angles = np.linspace(0, 2 * np.pi, 720, endpoint=False)\n    radii = np.arange(0.38 * width, 0.52 * width, 1.0)\n    cx, cy = width / 2, height / 2\n    radius, rms, inlier_fraction = 0.0, np.inf, 0.0\n    for _ in range(passes):\n        points = _limb_points(gray, cx, cy, radii, angles)\n        # Start from the median radius so prominences cannot drag the first fit.\n        first = np.hypot(points[:, 0] - cx, points[:, 1] - cy)\n        seed = np.abs(first - np.median(first)) <= 6.0\n        cx, cy, radius = _fit_circle(points[seed] if seed.sum() >= 100 else points)\n        for _ in range(3):\n            distance = np.hypot(points[:, 0] - cx, points[:, 1] - cy)\n            inliers = np.abs(distance - radius) <= 4.0\n            if inliers.sum() < 100:\n                break\n            cx, cy, radius = _fit_circle(points[inliers])\n        distance = np.hypot(points[:, 0] - cx, points[:, 1] - cy)\n        inliers = np.abs(distance - radius) <= 4.0\n        inlier_fraction = float(inliers.mean())\n        rms = float(np.sqrt(np.mean((distance[inliers] - radius) ** 2))) if inliers.any() else np.inf\n        radii = np.arange(max(radius - 60, 0.3 * width), radius + 60, 1.0)\n    ok = (0.38 * width <= radius <= 0.52 * width and abs(cx - width / 2) <= 0.15 * width\n          and abs(cy - height / 2) <= 0.15 * height and inlier_fraction >= 0.6 and rms <= 2.5)\n    return float(cx), float(cy), float(radius), bool(ok)\n\n\ndef feature_channels(normalized, outside):\n    """Three views of one normalized frame: the frame, a denoised copy, a dark-ridge map.\n\n    The ridge map is a black top-hat (dark structures narrower than the kernel), lightly\n    blurred; it is computed at half size and scaled back up because it is smooth.\n    """\n    height, width = normalized.shape\n    smooth = cv2.GaussianBlur(normalized, (0, 0), SMOOTH_SIGMA)\n    half = cv2.resize(normalized, (width // 2, height // 2), interpolation=cv2.INTER_AREA)\n    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (TOPHAT_KERNEL, TOPHAT_KERNEL))\n    ridge = cv2.morphologyEx(half, cv2.MORPH_BLACKHAT, kernel).astype(np.float32)\n    ridge = cv2.GaussianBlur(ridge, (0, 0), TOPHAT_SIGMA)\n    ridge = cv2.resize(ridge, (width, height), interpolation=cv2.INTER_LINEAR)\n    ridge = np.clip(ridge * RIDGE_GAIN, 0, 255).astype(np.uint8)\n    smooth[outside] = OUTPUT_CENTER\n    ridge[outside] = 0\n    return np.stack([normalized, smooth, ridge], axis=-1)\n\n\ndef normalize_disk(gray, margin=10, features=False):\n    """Radially flat-field the disk, z-score it, and replace everything beyond the rim.\n\n    Pixels farther than radius + margin from the fitted center become the neutral\n    value 128, as does the whole frame\'s fallback when no disk can be fitted.\n    Returns the uint8 image and a dict of the fit. With features=True the image has three\n    channels (see feature_channels).\n    """\n    gray = np.asarray(gray, dtype=np.float32)\n    height, width = gray.shape\n    cx, cy, radius, ok = fit_disk(gray)\n    info = dict(cx=cx, cy=cy, radius=radius, margin=margin, disk_found=ok)\n    yy, xx = np.mgrid[:height, :width]\n    if not ok:\n        center, spread = float(np.median(gray)), max(1.4826 * float(np.median(np.abs(gray - np.median(gray)))), 1.0)\n        z = (gray - center) / spread\n        flat = np.clip(OUTPUT_CENTER + OUTPUT_SCALE * z, 0, 255).astype(np.uint8)\n        if features:\n            flat = feature_channels(flat, np.zeros(flat.shape, dtype=bool))\n        return flat, info\n    r = np.hypot(xx - cx, yy - cy).astype(np.float32)\n    # The profile spans the safety margin too, so the rim step is flattened, not clipped.\n    inside = r <= radius + margin\n    bins = int(np.ceil((radius + margin) / BIN_PIXELS)) + 1\n    index = np.minimum((r / BIN_PIXELS).astype(np.int32), bins - 1)\n    flat_index, flat_value = index[inside], gray[inside]\n    order = np.argsort(flat_index, kind="stable")\n    sorted_index, sorted_value = flat_index[order], flat_value[order]\n    edges = np.searchsorted(sorted_index, np.arange(bins + 1))\n    profile = np.full(bins, np.nan, dtype=np.float32)\n    for b in range(bins):\n        if edges[b + 1] - edges[b] >= 200:\n            profile[b] = np.median(sorted_value[edges[b]:edges[b + 1]])\n    known = ~np.isnan(profile)\n    profile = np.interp(np.arange(bins), np.nonzero(known)[0], profile[known]).astype(np.float32)\n    residual = gray - profile[index]\n    # The limb ring is not perfectly circular, so near the rim use a per-sector profile.\n    zone = inside & (r > radius - RIM_ZONE)\n    first = int((radius - RIM_ZONE) / BIN_PIXELS)\n    depth = bins - first\n    angle = np.arctan2(yy - cy, xx - cx)\n    sector = np.minimum(((angle + np.pi) / (2 * np.pi) * SECTORS).astype(np.int32), SECTORS - 1)\n    key = (sector * depth + np.clip(index - first, 0, depth - 1))[zone]\n    order = np.argsort(key, kind="stable")\n    sorted_key, sorted_value = key[order], gray[zone][order]\n    cuts = np.searchsorted(sorted_key, np.arange(SECTORS * depth + 1))\n    local = np.full(SECTORS * depth, np.nan, dtype=np.float32)\n    for k in range(SECTORS * depth):\n        if cuts[k + 1] - cuts[k] >= MIN_BIN_SAMPLES:\n            local[k] = np.median(sorted_value[cuts[k]:cuts[k + 1]])\n    local_value = local[sector * depth + np.clip(index - first, 0, depth - 1)]\n    use = zone & ~np.isnan(local_value)\n    residual[use] = (gray - local_value)[use]\n    core = r <= 0.95 * radius\n    center = float(np.median(residual[core]))\n    spread = max(1.4826 * float(np.median(np.abs(residual[core] - center))), 1.0)\n    z = (residual - center) / spread\n    output = np.clip(OUTPUT_CENTER + OUTPUT_SCALE * z, 0, 255)\n    output[r > radius + margin] = OUTPUT_CENTER\n    info.update(spread=spread)\n    output = output.astype(np.uint8)\n    if features:\n        output = feature_channels(output, r > radius + margin)\n    return output, info\n\n\ndef normalize_file(source, destination, margin=10, features=False):\n    """Normalize one frame to a lossless-enough grayscale JPEG; return its disk fit."""\n    from pathlib import Path\n    from PIL import Image\n\n    source, destination = Path(source), Path(destination)\n    with Image.open(source) as image:\n        if image.mode != "L":\n            raise ValueError(f"Expected a grayscale frame: {source}")\n        gray = np.asarray(image)\n    output, info = normalize_disk(gray, margin, features)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + ".tmp")\n    Image.fromarray(output).save(temporary, format="JPEG", quality=98, subsampling=0)\n    temporary.replace(destination)\n    info.update(name=source.name, shape=list(gray.shape))\n    return info\n')
(runtime_dir / "v6_crop_runtime.py").write_text('"""CPU crop caching and batched refiner inference for the standalone V6 notebook."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\n\ndef crop_bounds(mask, context=1.8, minimum=96):\n    ys, xs = np.where(mask)\n    if not len(xs):\n        raise ValueError("Cannot crop an empty instance.")\n    height, width = mask.shape\n    side = min(height, width, max(minimum, int(np.ceil(\n        max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))))\n    x = int(np.clip(round((xs.max() + xs.min() + 1 - side) / 2), 0, width - side))\n    y = int(np.clip(round((ys.max() + ys.min() + 1 - side) / 2), 0, height - side))\n    return x, y, x + side, y + side\n\n\ndef normalized_crop(image, bounds, size):\n    array = np.asarray(image.crop(bounds).resize(\n        (size, size), Image.Resampling.BILINEAR), dtype=np.float32) / 255\n    return np.repeat(((array - 0.5) / 0.5)[None], 3, axis=0).copy()\n\n\nclass CachedInstanceCrops(Dataset):\n    """Cache compact pre-augmentation uint8 image/mask pairs, shared across folds."""\n\n    def __init__(self, entries, cache_dir, size=256, context=1.8, minimum=96, augment=False):\n        self.size, self.context, self.minimum = size, context, minimum\n        self.augment = augment\n        directory = Path(cache_dir)\n        directory.mkdir(parents=True, exist_ok=True)\n        self.entries = []\n        for path, rle in entries:\n            path = Path(path)\n            stat = path.stat()\n            counts = rle["counts"]\n            if not isinstance(counts, bytes):\n                raise TypeError("Cached crop entries require compressed byte-count RLEs.")\n            identity = json.dumps([\n                str(path.resolve()), stat.st_size, stat.st_mtime_ns, list(rle["size"]),\n                size, context, minimum, "uint8-bilinear-nearest-v1",\n            ]).encode() + counts\n            key = hashlib.sha256(identity).hexdigest()\n            self.entries.append((path, rle, directory / f"{key}.npy"))\n\n    def __len__(self):\n        return len(self.entries)\n\n    def __getitem__(self, index):\n        path, rle, cached = self.entries[index]\n        if cached.exists():\n            pair = np.load(cached, allow_pickle=False)\n            if pair.shape != (2, self.size, self.size) or pair.dtype != np.uint8:\n                raise ValueError(f"Invalid cached crop: {cached}")\n        else:\n            mask = mask_utils.decode(rle)\n            bounds = crop_bounds(mask, self.context, self.minimum)\n            with Image.open(path) as image:\n                if image.mode != "L" or image.size != (mask.shape[1], mask.shape[0]):\n                    raise ValueError(f"Unexpected grayscale crop image: {path}")\n                pixels = np.asarray(image.crop(bounds).resize(\n                    (self.size, self.size), Image.Resampling.BILINEAR), dtype=np.uint8)\n            x0, y0, x1, y1 = bounds\n            target = np.asarray(Image.fromarray(mask[y0:y1, x0:x1]).resize(\n                (self.size, self.size), Image.Resampling.NEAREST), dtype=np.uint8)\n            pair = np.stack([pixels, target])\n            temporary = cached.with_name(f"{cached.stem}.{os.getpid()}.tmp.npy")\n            try:\n                np.save(temporary, pair, allow_pickle=False)\n                temporary.replace(cached)\n            finally:\n                if temporary.exists():\n                    temporary.unlink()\n        if not np.isin(pair[1], [0, 1]).all():\n            raise ValueError(f"Nonbinary cached crop target: {cached}")\n        x = np.repeat(((pair[0].astype(np.float32) / 255 - 0.5) / 0.5)[None], 3, axis=0)\n        y = pair[1].astype(np.float32)[None]\n        if self.augment:\n            turns = int(torch.randint(4, ()).item())\n            x, y = np.rot90(x, turns, axes=(1, 2)), np.rot90(y, turns, axes=(1, 2))\n            if torch.rand(()).item() < 0.5:\n                x, y = np.flip(x, axis=2), np.flip(y, axis=2)\n        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))\n\n\ndef loader_options(workers, cuda):\n    if type(workers) is not int or workers < 0:\n        raise ValueError("DataLoader worker count must be a nonnegative integer.")\n    options = dict(num_workers=workers, pin_memory=cuda)\n    if workers:\n        options.update(persistent_workers=True, prefetch_factor=2,\n                       multiprocessing_context="spawn")\n    return options\n\n\ndef parallel_refiner(model, device_ids):\n    if not device_ids:\n        return model\n    if len(device_ids) != len(set(device_ids)):\n        raise ValueError("Refiner device IDs must be unique.")\n    if len(device_ids) > 1:\n        return torch.nn.DataParallel(model, device_ids=device_ids, output_device=device_ids[0])\n    return model\n\n\ndef base_refiner(model):\n    return model.module if isinstance(model, torch.nn.DataParallel) else model\n\n\n@torch.inference_mode()\ndef refine_proposals(path, masks, refiner, device, shape, batch_size=16,\n                     crop_size=256, context=1.8, minimum=96):\n    if type(batch_size) is not int or batch_size < 1:\n        raise ValueError("Crop inference batch size must be a positive integer.")\n    if masks.ndim != 3 or masks.shape[1:] != tuple(shape) or not np.isfinite(masks).all():\n        raise ValueError("YOLO masks are not finite full-resolution masks.")\n    proposals = []\n    with Image.open(path) as image:\n        if image.mode != "L" or image.size != (shape[1], shape[0]):\n            raise ValueError(f"Unexpected grayscale inference image: {path}")\n        for start in range(0, len(masks), batch_size):\n            batch, metadata = [], []\n            for mask in masks[start:start + batch_size]:\n                coarse = (mask >= .5).astype(np.uint8)\n                if not coarse.any():\n                    print("WARNING: YOLO returned an empty mask; omitted:", path.name)\n                    continue\n                bounds = crop_bounds(coarse, context, minimum)\n                batch.append(normalized_crop(image, bounds, crop_size))\n                metadata.append((mask_utils.encode(np.asfortranarray(coarse)), bounds))\n            if not batch:\n                continue\n            tensor = torch.from_numpy(np.stack(batch)).to(device)\n            probabilities = refiner(tensor).sigmoid()[:, 0].cpu().numpy()\n            if probabilities.shape != (len(batch), crop_size, crop_size):\n                raise ValueError("Refiner output does not match crop batch shape.")\n            if not np.isfinite(probabilities).all():\n                raise FloatingPointError("Nonfinite refinement probability.")\n            proposals.extend(dict(coarse=rle, bounds=bounds, probability=probability)\n                             for (rle, bounds), probability in zip(metadata, probabilities))\n    return proposals\n')
(runtime_dir / "v7_yolo_runtime.py").write_text('"""YOLO-only five-fold execution, resumable batch inference, and audited export."""\n\nimport csv\nimport gc\nimport hashlib\nimport json\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\nimport torch\nimport yaml\nfrom ultralytics import YOLO\n\nfrom grouped_yolo_cv import (\n    ENSEMBLE_POLICY, cv_digest, fold_records, fuse_fold_instances,\n    validate_fold_result,\n)\nfrom submission_masks import (\n    SUBMISSION_FIELDS, disjoint_instances, validate_disjoint, validate_submission_rows,\n)\n\n\ndef file_hash(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))\n    temporary.replace(path)\n\n\ndef require_identity(path, identity):\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != identity:\n            raise ValueError(f"Provenance mismatch: {path}. Use a new run directory.")\n    else:\n        atomic_json(path, identity)\n\n\ndef prepare_dataset(ns, plan, directory):\n    """Keep distinct annotator targets, sharing their physical source JPEGs."""\n    data = directory / "yolo_dataset"\n    records = fold_records(plan, ns["records_by_name"])\n    height, width = ns["FULL_SHAPE"]\n    for phase, label in [("train", "train"), ("tune", "val")]:\n        images, labels = data / "images" / label, data / "labels" / label\n        images.mkdir(parents=True, exist_ok=True)\n        labels.mkdir(parents=True, exist_ok=True)\n        for index, record in enumerate(records[phase]):\n            stem = f\'{Path(record["file_name"]).stem}__annotation{index}\'\n            source = ns["train_lookup"][record["file_name"]].resolve()\n            destination = images / f"{stem}.jpeg"\n            if not destination.exists():\n                destination.symlink_to(source)\n            elif destination.resolve() != source:\n                raise ValueError(f"Training image link differs: {destination}")\n            lines = []\n            for annotation in ns["annotations_by_id"][record["id"]]:\n                polygons = annotation["segmentation"]\n                if not isinstance(polygons, list) or len(polygons) != 1:\n                    raise ValueError("YOLO conversion requires one polygon per instance.")\n                points = np.asarray(polygons[0], dtype=np.float64)\n                if points.size < 6 or points.size % 2 or not np.isfinite(points).all():\n                    raise ValueError(f\'Invalid polygon {annotation["id"]}\')\n                points = points.reshape(-1, 2)\n                if np.any(points < 0) or np.any(points > [width, height]):\n                    raise ValueError(f\'Out-of-bounds polygon {annotation["id"]}\')\n                lines.append("0 " + " ".join(\n                    f"{value:.8f}" for value in (points / [width, height]).ravel()))\n            (labels / f"{stem}.txt").write_text("\\n".join(lines) + ("\\n" if lines else ""))\n    path = data / "data.yaml"\n    path.write_text(yaml.safe_dump(dict(\n        path=str(data.resolve()), train="images/train", val="images/val",\n        names={0: "filament"},\n    )))\n    return path\n\n\ndef train_detector(ns, plan, shared_plan):\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    directory.mkdir(parents=True, exist_ok=True)\n    identity = dict(plan_sha256=cv_digest(shared_plan), fold=plan["fold"])\n    require_identity(directory / "fold_identity.json", identity)\n    checkpoint, provenance = directory / "yolo_best.pt", directory / "yolo_provenance.json"\n    if provenance.exists():\n        saved = json.loads(provenance.read_text())\n        if saved["identity"] != identity or saved["sha256"] != file_hash(checkpoint):\n            raise ValueError("Saved detector checkpoint provenance mismatch.")\n        print(f"Fold {plan[\'fold\']}: reusing verified detector", flush=True)\n        return YOLO(str(checkpoint))\n    if checkpoint.exists():\n        raise ValueError("Checkpoint publication was interrupted; preserve it and inspect provenance.")\n    if not torch.cuda.is_available() and not ns["SMOKE_TEST"]:\n        raise RuntimeError("Enable a GPU before full training.")\n    settings = ns["V7"]\n    data = prepare_dataset(ns, plan, directory)\n    run = directory / "detector_runs" / "yolo"\n    last, best = run / "weights" / "last.pt", run / "weights" / "best.pt"\n    completed = False\n    if last.exists():\n        # Only read checkpoints generated by this identity-checked local run.\n        saved = torch.load(last, map_location="cpu", weights_only=False)\n        completed = saved["epoch"] < 0 or saved["epoch"] + 1 >= settings["epochs"]\n        del saved\n    started = time.monotonic()\n    if not completed:\n        if last.exists():\n            print(f"Fold {plan[\'fold\']}: resuming last completed YOLO epoch", flush=True)\n            model = YOLO(str(last))\n            model.train(resume=True, device=ns["YOLO_TRAIN_DEVICE"], workers=settings["workers"])\n        else:\n            print(f"Fold {plan[\'fold\']}: starting YOLO (no saved epoch)", flush=True)\n            model = YOLO(settings["weights"])\n            model.train(\n                data=str(data), epochs=settings["epochs"], imgsz=settings["image_size"],\n                batch=settings["batch_size"], device=ns["YOLO_TRAIN_DEVICE"],\n                workers=settings["workers"], patience=settings["patience"], time=None,\n                optimizer="AdamW", lr0=settings["learning_rate"], momentum=0.9,\n                weight_decay=0.0005, warmup_bias_lr=0.0, seed=ns["CFG"]["seed"],\n                single_cls=True, mask_ratio=2, overlap_mask=False,\n                hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, fliplr=0.5, flipud=0.0,\n                mosaic=0.3, degrees=10.0, scale=0.3, close_mosaic=5,\n                cache=False, save=True, save_period=1, plots=False,\n                project=str(run.parent), name=run.name, exist_ok=True,\n            )\n        del model\n        gc.collect()\n    if not best.is_file():\n        raise FileNotFoundError(f"No best detector checkpoint: {best}")\n    temporary = checkpoint.with_suffix(".tmp.pt")\n    shutil.copyfile(best, temporary)\n    temporary.replace(checkpoint)\n    atomic_json(provenance, dict(\n        identity=identity, sha256=file_hash(checkpoint), settings=settings,\n        latest_invocation_seconds=time.monotonic() - started,\n        note="Invocation time excludes earlier interrupted attempts; see YOLO results.csv.",\n    ))\n    return YOLO(str(checkpoint))\n\n\ndef encode_masks(instances):\n    return [dict(size=list(rle["size"]), counts=rle["counts"].decode("ascii"))\n            for rle in instances]\n\n\ndef decode_masks(items):\n    return [dict(size=item["size"], counts=item["counts"].encode("ascii")) for item in items]\n\n\ndef image_manifest(paths):\n    if not paths or len({p.stem for p in paths}) != len(paths):\n        raise ValueError("Prediction image IDs must be nonempty and unique.")\n    return [dict(name=p.name, sha256=file_hash(p)) for p in paths]\n\n\ndef predict_cached(ns, detector, paths, parameters, cache_path, identity):\n    """Save each completed inference batch; restart only missing images."""\n    paths = list(paths)\n    shape = ns["FULL_SHAPE"]\n    expected = {p.stem for p in paths}\n    cache_identity = dict(\n        model=identity, parameters=parameters,\n        images_sha256=cv_digest(image_manifest(paths)), shape=list(shape),\n    )\n    cache_path = Path(cache_path)\n    if cache_path.exists():\n        cache = json.loads(cache_path.read_text())\n        if cache["identity"] != cache_identity:\n            raise ValueError(f"Stale prediction cache: {cache_path}")\n    else:\n        cache = dict(identity=cache_identity, predictions={})\n    predictions = cache["predictions"]\n    if not set(predictions) <= expected:\n        raise ValueError("Prediction cache contains unknown image IDs.")\n    for items in predictions.values():\n        validate_disjoint(decode_masks(items), shape)\n    missing = [p for p in paths if p.stem not in predictions]\n    batch_size = ns["V7"]["inference_batch"]\n    if type(batch_size) is not int or batch_size < 1:\n        raise ValueError("inference_batch must be a positive integer.")\n    print(f"{cache_path.name}: {len(predictions)}/{len(paths)} saved", flush=True)\n    for start in range(0, len(missing), batch_size):\n        batch = missing[start:start + batch_size]\n        results = detector.predict(\n            source=[str(p) for p in batch], batch=batch_size,\n            imgsz=ns["V7"]["image_size"], conf=parameters["confidence"],\n            device=ns["YOLO_INFERENCE_DEVICE"], agnostic_nms=True, retina_masks=True,\n            verbose=False, stream=False,\n        )\n        if len(results) != len(batch):\n            raise ValueError("Detector returned an unexpected batch length.")\n        for path, result in zip(batch, results):\n            if Path(result.path).resolve() != path.resolve() or tuple(result.orig_shape) != tuple(shape):\n                raise ValueError("Detector result path/dimensions differ from its input.")\n            instances = []\n            if result.masks is not None:\n                masks = result.masks.data.cpu().numpy()\n                if masks.ndim != 3 or tuple(masks.shape[1:]) != tuple(shape):\n                    raise ValueError("Detector masks must retain original image dimensions.")\n                if not np.isin(masks, [0, 1]).all():\n                    raise ValueError("Detector masks must be finite and binary.")\n                instances = [mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))\n                             for mask in masks if mask.any()]\n            instances, _ = disjoint_instances(instances, shape, parameters["min_area"])\n            validate_disjoint(instances, shape)\n            predictions[path.stem] = encode_masks(instances)\n        atomic_json(cache_path, cache)\n        print(f"{cache_path.name}: {len(predictions)}/{len(paths)} saved", flush=True)\n    if set(predictions) != expected:\n        raise ValueError("Incomplete inference coverage.")\n    return predictions\n\n\ndef score_predictions(ns, names, predictions):\n    rows = []\n    for name in names:\n        instances = decode_masks(predictions[Path(name).stem])\n        scores = [ns["local_scores"](instances, ns["gt_rles"](record["id"]))\n                  for record in ns["records_by_name"][name]]\n        if not scores:\n            raise ValueError(f"No annotator sets: {name}")\n        rows.append(dict(\n            filename=name, annotator_sets=len(scores), instances=len(instances),\n            **{key: float(np.mean([score[key] for score in scores])) for key in scores[0]},\n        ))\n    if not rows:\n        raise ValueError("No evaluation images.")\n    means = {key: float(np.mean([row[key] for row in rows]))\n             for key in rows[0] if key not in ["filename", "annotator_sets"]}\n    return means, rows\n\n\ndef calibrate(ns, detector, plan, identity):\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    path = directory / "calibration.json"\n    if path.exists():\n        saved = json.loads(path.read_text())\n        if saved["identity"] != identity:\n            raise ValueError("Calibration provenance mismatch.")\n        return saved\n    names = plan["splits"]["tune"][:ns["V7"]["tuning_images"]]\n    trials = []\n    for confidence in ns["V7"]["confidences"]:\n        for min_area in ns["V7"]["min_areas"]:\n            parameters = dict(mode="yolo", confidence=confidence, min_area=min_area)\n            predictions = predict_cached(\n                ns, detector, [ns["train_lookup"][name] for name in names], parameters,\n                directory / f"tune_{confidence}_{min_area}.json", identity,\n            )\n            means, _ = score_predictions(ns, names, predictions)\n            trials.append(dict(parameters=parameters, scores=means))\n    selected = max(trials, key=lambda trial: trial["scores"]["pq"])\n    result = dict(identity=identity, selected=selected, trials=trials, tuning_names=names)\n    atomic_json(path, result)\n    print("TUNING ONLY, not independent performance:", selected, flush=True)\n    return result\n\n\ndef fold_model_identity(ns, plan, shared_plan):\n    checkpoint = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}" / "yolo_best.pt"\n    return dict(plan_sha256=cv_digest(shared_plan), fold=plan["fold"],\n                checkpoint_sha256=file_hash(checkpoint))\n\n\ndef run_fold(ns, plan, shared_plan):\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    path = directory / "fold_result.json"\n    if path.exists():\n        result = json.loads(path.read_text())\n        validate_fold_result(result, plan, cv_digest(shared_plan))\n        identity = fold_model_identity(ns, plan, shared_plan)\n        if result["checkpoint_hashes"] != {"yolo_best.pt": identity["checkpoint_sha256"]}:\n            raise ValueError("Completed fold checkpoint changed.")\n        print(f"Fold {plan[\'fold\']}: reusing completed outer evaluation", flush=True)\n        return result\n    detector = train_detector(ns, plan, shared_plan)\n    try:\n        identity = fold_model_identity(ns, plan, shared_plan)\n        calibration = calibrate(ns, detector, plan, identity)\n        parameters = calibration["selected"]["parameters"]\n        names = plan["splits"]["holdout"]\n        predictions = predict_cached(\n            ns, detector, [ns["train_lookup"][name] for name in names], parameters,\n            directory / "outer_predictions.json", identity,\n        )\n        means, rows = score_predictions(ns, names, predictions)\n        result = dict(\n            fold=plan["fold"], plan_sha256=cv_digest(shared_plan), status="complete",\n            smoke_test=ns["SMOKE_TEST"],\n            checkpoint_hashes={"yolo_best.pt": identity["checkpoint_sha256"]},\n            selected_parameters=parameters, inner_selected_scores=calibration["selected"]["scores"],\n            outer_mean=means, outer_rows=rows,\n        )\n        validate_fold_result(result, plan, cv_digest(shared_plan))\n        atomic_json(path, result)\n        return result\n    finally:\n        del detector\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n\ndef write_submission(ns, shared_plan, predictions, results):\n    """Validate the exact CSV read back from disk and publish its checksum last."""\n    if ns["SMOKE_TEST"] or shared_plan["smoke_test"]:\n        raise ValueError("Smoke experiments cannot export competition submissions.")\n    root, shape = ns["CV_ROOT"], ns["FULL_SHAPE"]\n    paths = ns["test_paths"]\n    names = {path.stem for path in paths}\n    if len(predictions) != 5 or any(set(fold) != names for fold in predictions):\n        raise ValueError("Export requires complete test coverage from exactly five models.")\n    rows = []\n    for index, path in enumerate(paths):\n        per_fold = [decode_masks(fold[path.stem]) for fold in predictions]\n        instances = fuse_fold_instances(per_fold, shape)\n        instances, _ = disjoint_instances(instances, shape, ENSEMBLE_POLICY["min_area"])\n        validate_disjoint(instances, shape)\n        rows.extend(dict(filament_id=f"{path.stem}_{i}", segmentation_rle=rle["counts"].decode("ascii"))\n                    for i, rle in enumerate(instances, 1))\n        print(f"Fusion: {index + 1}/{len(paths)}", flush=True)\n    validate_submission_rows(rows, names, shape)\n    destination = root / "submission_V7_yolo_only.csv"\n    temporary = destination.with_suffix(".tmp.csv")\n    with temporary.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=SUBMISSION_FIELDS)\n        writer.writeheader()\n        writer.writerows(rows)\n    with temporary.open(newline="") as stream:\n        reader = csv.DictReader(stream)\n        if reader.fieldnames != SUBMISSION_FIELDS:\n            raise ValueError("Written CSV columns differ.")\n        readback = list(reader)\n    if readback != rows:\n        raise ValueError("Written CSV differs from generated predictions.")\n    audit = validate_submission_rows(readback, names, shape)\n    digest = file_hash(temporary)\n    temporary.replace(destination)\n    atomic_json(root / "ensemble_audit.json", dict(\n        run_name=root.name, architecture="YOLO-only; no crop refiner",\n        plan_sha256=cv_digest(shared_plan), policy=ENSEMBLE_POLICY,\n        test_manifest_sha256=cv_digest(image_manifest(paths)),\n        folds=[dict(fold=r["fold"], hashes=r["checkpoint_hashes"],\n                    parameters=r["selected_parameters"]) for r in results],\n        validation=audit, submission_filename=destination.name, submission_sha256=digest,\n    ))\n    if file_hash(destination) != digest:\n        raise ValueError("Final submission checksum mismatch.")\n    print("FINAL V7 CSV VERIFIED:", destination, "\\nSHA256:", digest, "\\nVALIDATION:", audit,\n          "\\nNo upload performed. Match this hash before manual submission.", flush=True)\n    return destination\n\n\ndef export_ensemble(ns, plans, shared_plan):\n    if ns["SMOKE_TEST"] or shared_plan["smoke_test"]:\n        raise ValueError("Smoke experiments cannot export competition submissions.")\n    if sorted(plan["fold"] for plan in plans) != list(range(5)):\n        raise ValueError("Five unique fold plans are required.")\n    plans = sorted(plans, key=lambda plan: plan["fold"])\n    results, predictions = [], []\n    for plan in plans:\n        directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n        result = json.loads((directory / "fold_result.json").read_text())\n        validate_fold_result(result, plan, cv_digest(shared_plan))\n        identity = fold_model_identity(ns, plan, shared_plan)\n        if result["checkpoint_hashes"] != {"yolo_best.pt": identity["checkpoint_sha256"]}:\n            raise ValueError("Export checkpoint differs from evaluated model.")\n        calibration = json.loads((directory / "calibration.json").read_text())\n        if (calibration["identity"] != identity\n                or calibration["selected"]["parameters"] != result["selected_parameters"]):\n            raise ValueError("Export calibration differs from evaluated settings.")\n        detector = YOLO(str(directory / "yolo_best.pt"))\n        try:\n            predictions.append(predict_cached(\n                ns, detector, ns["test_paths"], result["selected_parameters"],\n                directory / "test_predictions.json", identity,\n            ))\n            results.append(result)\n        finally:\n            del detector\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n    return write_submission(ns, shared_plan, predictions, results)\n')
(runtime_dir / "v8_runtime.py").write_text('"""V8: five-fold YOLO + crop-refiner execution, resumable inference, and audited export."""\n\nimport csv\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport time\nfrom collections import defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\nimport scipy.ndimage as ndi\nimport segmentation_models_pytorch as smp\nimport torch\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom scipy.optimize import linear_sum_assignment\nfrom torch.utils.data import DataLoader, Dataset\nfrom ultralytics import YOLO\n\nfrom grouped_yolo_cv import cv_digest, fold_records, validate_fold_result\nfrom solar_disk import normalize_file\nfrom submission_masks import (\n    SUBMISSION_FIELDS, disjoint_instances, validate_disjoint, validate_submission_rows,\n)\nfrom v6_crop_runtime import (\n    CachedInstanceCrops, base_refiner, crop_bounds, loader_options, normalized_crop,\n    parallel_refiner,\n)\nfrom v7_yolo_runtime import (\n    atomic_json, decode_masks, encode_masks, file_hash, image_manifest,\n    prepare_dataset, require_identity, score_predictions,\n)\n\nSAVE_EVERY = 8\nMODES = ("yolo", "refined", "gated", "zoom")\nYOLO_AUGMENT_DEFAULTS = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, fliplr=0.5, flipud=0.0,\n                             mosaic=0.3, degrees=10.0, scale=0.3, close_mosaic=5)\n\n\ndef yolo_augmentation(settings):\n    """Detector augmentation: V8-V10 defaults, overridable through settings["yolo_augment"].\n\n    The solar disk is centered in the frame and everything outside it is neutral gray, so\n    a rotation about the frame center is a rotation of the Sun about its own center.\n    """\n    custom = settings.get("yolo_augment", {})\n    unknown = set(custom) - set(YOLO_AUGMENT_DEFAULTS)\n    if unknown:\n        raise ValueError(f"Unknown YOLO augmentation settings: {sorted(unknown)}")\n    return {**YOLO_AUGMENT_DEFAULTS, **custom}\n\n\ndef rotate_batch(x, y, angles=None, scales=None, shifts=None):\n    """Rotate (and optionally zoom and shift) each (image, mask) crop about its center.\n\n    scales below 1 zoom in; shifts are fractions of the half-width. Images use reflection\n    padding; masks use zero padding and are re-binarized.\n    """\n    count = x.shape[0]\n    if angles is None:\n        angles = torch.rand(count, device=x.device) * 2 * math.pi\n    angles = angles.to(x.dtype)\n    scale = torch.ones_like(angles) if scales is None else scales.to(x.dtype)\n    shift = torch.zeros(count, 2, dtype=x.dtype, device=x.device) if shifts is None else shifts.to(x.dtype)\n    cos, sin = angles.cos() * scale, angles.sin() * scale\n    matrix = torch.stack([torch.stack([cos, -sin, shift[:, 0]], dim=1),\n                          torch.stack([sin, cos, shift[:, 1]], dim=1)], dim=1)\n    grid = F.affine_grid(matrix, list(x.shape), align_corners=False)\n    rotated = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)\n    mask = F.grid_sample(y, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n    return rotated, (mask >= 0.5).to(y.dtype)\n\n\ndef jitter_batch(x, y):\n    """Random rotation plus the zoom and shift an imperfect detector box would cause."""\n    count = x.shape[0]\n    scales = 0.75 + 0.4 * torch.rand(count, device=x.device)\n    shifts = (torch.rand(count, 2, device=x.device) - 0.5) * 0.16\n    return rotate_batch(x, y, scales=scales, shifts=shifts)\n\n\ndef simulate_coarse(y, generator=None):\n    """Detector-like masks for a batch of ground-truth crop masks (N, 1, H, W).\n\n    Half of the samples get a rough mask (blocky edges, shifted, grown or shrunk, sometimes\n    only part of the filament) like YOLO\'s; the rest get a fine one like a first refinement\n    pass. A fixed generator makes the validation inputs identical every epoch.\n    """\n    count, _, height, width = y.shape\n    draws = torch.rand(count, 9, generator=generator).tolist()\n    out = torch.empty_like(y)\n    rows = torch.arange(height, device=y.device, dtype=y.dtype)[:, None].expand(height, width)\n    columns = torch.arange(width, device=y.device, dtype=y.dtype)[None, :].expand(height, width)\n    for index, u in enumerate(draws):\n        truth = y[index:index + 1]\n        fine = u[0] < 0.4\n        cell = (2, 3, 4)[int(u[1] * 3)] if fine else (6, 8, 10, 14)[int(u[1] * 4)]\n        pooled = F.avg_pool2d(truth, cell, ceil_mode=True)\n        pooled = F.interpolate(pooled, size=(height, width), mode="bilinear", align_corners=False)\n        mask = (pooled >= 0.35 + 0.3 * u[2]).to(y.dtype)\n        reach = 2 if fine else 6\n        mask = torch.roll(mask, shifts=(int(round((u[3] - 0.5) * 2 * reach)),\n                                         int(round((u[4] - 0.5) * 2 * reach))), dims=(2, 3))\n        radius = int(round((u[5] - 0.5) * (2 if fine else 7)))\n        if radius > 0:\n            mask = F.max_pool2d(mask, 2 * radius + 1, 1, radius)\n        elif radius < 0:\n            mask = 1 - F.max_pool2d(1 - mask, 2 * -radius + 1, 1, -radius)\n        if not fine and u[6] < 0.15 and mask.sum() > 0:\n            angle = u[7] * 2 * math.pi\n            projection = math.cos(angle) * columns + math.sin(angle) * rows\n            values = projection[mask[0, 0] > 0]\n            keep = projection <= torch.quantile(values, 0.35 + 0.4 * u[8])\n            mask = mask * keep[None, None].to(y.dtype)\n        out[index] = mask[0] if mask.sum() >= 4 else truth[0]\n    return out\n\n\ndef prepare_images(ns):\n    """Write disk-normalized copies of every train/test frame and return their lookups.\n\n    Geometry is unchanged (still 2048 x 2048), so labels, crop windows and the CSV\n    coordinates are identical; only pixel values differ.\n    """\n    settings, root = ns["V8"], ns["SCRATCH_DIR"] / "_preprocessed"\n    jobs = ([(path, root / "train" / name) for name, path in sorted(ns["train_lookup"].items())]\n            + [(path, root / "test" / path.name) for path in ns["test_paths"]])\n    started = time.monotonic()\n    print(f"Disk normalization: starting {len(jobs)} frames", flush=True)\n    margin, fits = settings["disk_margin"], []\n    features = bool(settings.get("feature_channels", False))\n    # Threads, not processes: numpy/scipy release the GIL, and notebooks cannot hang on spawn.\n    with ThreadPoolExecutor(max_workers=settings["preprocess_workers"]) as pool:\n        for done, fit in enumerate(pool.map(lambda job: normalize_file(job[0], job[1], margin, features), jobs), 1):\n            fits.append(fit)\n            if done % 100 == 0 or done == len(jobs):\n                print(f"Disk normalization: {done}/{len(jobs)} frames "\n                      f"({time.monotonic() - started:.0f}s)", flush=True)\n    missing = [fit["name"] for fit in fits if not fit["disk_found"]]\n    if len(missing) > 0.01 * len(fits):\n        raise ValueError(f"Solar disk not found in {len(missing)} frames: {missing[:5]}")\n    radii = np.array([fit["radius"] for fit in fits if fit["disk_found"]])\n    report = dict(frames=len(fits), disk_not_found=missing, radius_min=float(radii.min()),\n                  radius_median=float(np.median(radii)), radius_max=float(radii.max()),\n                  margin=settings["disk_margin"], seconds=time.monotonic() - started)\n    atomic_json(ns["CV_ROOT"] / "disk_fits.json", dict(report=report, fits=fits))\n    print("Disk normalization:", report, flush=True)\n    train_lookup = {name: destination for (_, destination), name in\n                    zip(jobs[:len(ns["train_lookup"])], sorted(ns["train_lookup"]))}\n    test_paths = [destination for _, destination in jobs[len(ns["train_lookup"]):]]\n    return train_lookup, test_paths, report\n\n\ndef normalized_crop_rgb(image, bounds, size):\n    array = np.asarray(image.crop(bounds).resize((size, size), Image.Resampling.BILINEAR),\n                       dtype=np.float32) / 255\n    return ((array - 0.5) / 0.5).transpose(2, 0, 1).copy()\n\n\ndef crop_tensor(image, bounds, size):\n    """Refiner input for one window: three real channels, or one gray channel repeated."""\n    return (normalized_crop_rgb(image, bounds, size) if image.mode == "RGB"\n            else normalized_crop(image, bounds, size))\n\n\ndef images_are_rgb(ns):\n    with Image.open(next(iter(ns["train_lookup"].values()))) as image:\n        return image.mode == "RGB"\n\n\nclass CachedFeatureCrops(Dataset):\n    """Instance crops of three-channel frames, cached as uint8 (3 image + 1 mask) arrays."""\n\n    def __init__(self, entries, cache_dir, size=256, context=1.8, minimum=96, augment=False):\n        self.size, self.context, self.minimum, self.augment = size, context, minimum, augment\n        directory = Path(cache_dir)\n        directory.mkdir(parents=True, exist_ok=True)\n        self.entries = []\n        for path, rle in entries:\n            path = Path(path)\n            stat = path.stat()\n            counts = rle["counts"]\n            if not isinstance(counts, bytes):\n                raise TypeError("Cached crop entries require compressed byte-count RLEs.")\n            identity = json.dumps([str(path.resolve()), stat.st_size, stat.st_mtime_ns,\n                                   list(rle["size"]), size, context, minimum,\n                                   "rgb-uint8-bilinear-nearest-v1"]).encode() + counts\n            self.entries.append((path, rle, directory / f"{hashlib.sha256(identity).hexdigest()}.npy"))\n\n    def __len__(self):\n        return len(self.entries)\n\n    def _write(self, index, image):\n        path, rle, cached = self.entries[index]\n        mask = mask_utils.decode(rle)\n        x0, y0, x1, y1 = crop_bounds(mask, self.context, self.minimum)\n        pixels = np.asarray(image.crop((x0, y0, x1, y1)).resize(\n            (self.size, self.size), Image.Resampling.BILINEAR), dtype=np.uint8)\n        target = np.asarray(Image.fromarray(mask[y0:y1, x0:x1]).resize(\n            (self.size, self.size), Image.Resampling.NEAREST), dtype=np.uint8)\n        pair = np.concatenate([pixels.transpose(2, 0, 1), target[None]], axis=0)\n        temporary = cached.with_name(f"{cached.stem}.{os.getpid()}.{index}.tmp.npy")\n        try:\n            np.save(temporary, pair, allow_pickle=False)\n            temporary.replace(cached)\n        finally:\n            if temporary.exists():\n                temporary.unlink()\n        return pair\n\n    def prewarm(self, threads=4):\n        """Decode each frame once and write all of its missing crops."""\n        groups = defaultdict(list)\n        for index, (path, _, cached) in enumerate(self.entries):\n            if not cached.exists():\n                groups[path].append(index)\n\n        def work(item):\n            path, indices = item\n            with Image.open(path) as image:\n                if image.mode != "RGB":\n                    raise ValueError(f"Expected a three-channel frame: {path}")\n                image.load()\n                for index in indices:\n                    self._write(index, image)\n\n        with ThreadPoolExecutor(max_workers=threads) as pool:\n            list(pool.map(work, groups.items()))\n\n    def __getitem__(self, index):\n        path, rle, cached = self.entries[index]\n        if cached.exists():\n            pair = np.load(cached, allow_pickle=False)\n        else:\n            with Image.open(path) as image:\n                pair = self._write(index, image.convert("RGB") if image.mode != "RGB" else image)\n        if pair.shape != (4, self.size, self.size) or pair.dtype != np.uint8:\n            raise ValueError(f"Invalid cached crop: {cached}")\n        x = (pair[:3].astype(np.float32) / 255 - 0.5) / 0.5\n        y = pair[3:4].astype(np.float32)\n        if self.augment:\n            turns = int(torch.randint(4, ()).item())\n            x, y = np.rot90(x, turns, axes=(1, 2)), np.rot90(y, turns, axes=(1, 2))\n            if torch.rand(()).item() < 0.5:\n                x, y = np.flip(x, axis=2), np.flip(y, axis=2)\n        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))\n\n\ndef window_mask(mask, bounds, size):\n    """A full-frame binary mask cut to a window and resized to the refiner\'s input size."""\n    x0, y0, x1, y1 = bounds\n    return np.asarray(Image.fromarray(mask[y0:y1, x0:x1]).resize(\n        (size, size), Image.Resampling.NEAREST), dtype=np.float32)\n\n\n@torch.inference_mode()\ndef refine_image(image, masks, refiner, device, shape, batch_size=16, crop_size=256,\n                 context=1.8, minimum=96, mask_channel=False):\n    """Crop every detected mask from `image`, run the refiner, return per-crop probabilities.\n\n    With mask_channel the detector\'s mask, cut to the same window, is the fourth input channel.\n    """\n    if type(batch_size) is not int or batch_size < 1:\n        raise ValueError("Crop inference batch size must be a positive integer.")\n    if masks.ndim != 3 or masks.shape[1:] != tuple(shape):\n        raise ValueError("Detector masks are not full-resolution masks.")\n    if image.size != (shape[1], shape[0]):\n        raise ValueError("Inference image does not match the mask resolution.")\n    proposals = []\n    for start in range(0, len(masks), batch_size):\n        batch, metadata = [], []\n        for mask in masks[start:start + batch_size]:\n            coarse = (mask >= 0.5).astype(np.uint8)\n            if not coarse.any():\n                continue\n            bounds = crop_bounds(coarse, context, minimum)\n            crop = crop_tensor(image, bounds, crop_size)\n            if mask_channel:\n                crop = np.concatenate([crop, window_mask(coarse, bounds, crop_size)[None]], axis=0)\n            batch.append(crop)\n            metadata.append((mask_utils.encode(np.asfortranarray(coarse)), bounds))\n        if not batch:\n            continue\n        tensor = torch.from_numpy(np.stack(batch)).to(device)\n        probabilities = refiner(tensor).sigmoid()[:, 0].cpu().numpy()\n        if probabilities.shape != (len(batch), crop_size, crop_size):\n            raise ValueError("Refiner output does not match crop batch shape.")\n        if not np.isfinite(probabilities).all():\n            raise FloatingPointError("Nonfinite refinement probability.")\n        proposals.extend(dict(coarse=rle, bounds=bounds, probability=probability)\n                         for (rle, bounds), probability in zip(metadata, probabilities))\n    return proposals\n\n\ndef segmentation_loss(logits, target):\n    """Equal-weight BCE and soft Dice, as in the scored V1-V5 U-Net losses."""\n    probabilities = logits.sigmoid()\n    intersection = (probabilities * target).sum(dim=(1, 2, 3))\n    denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))\n    dice = 1 - ((2 * intersection + 1) / (denominator + 1)).mean()\n    return 0.5 * F.binary_cross_entropy_with_logits(logits, target) + 0.5 * dice\n\n\ndef refiner_channels(settings):\n    """Three image channels, plus the detector mask as a fourth when enabled."""\n    return 4 if settings.get("refiner_mask_channel", False) else 3\n\n\ndef build_refiner(settings, pretrained):\n    return smp.Unet(\n        encoder_name=settings["refiner_encoder"],\n        encoder_weights="imagenet" if pretrained else None,\n        in_channels=refiner_channels(settings), classes=1, activation=None,\n    )\n\n\ndef load_refiner(ns, path):\n    saved = torch.load(path, weights_only=True, map_location="cpu")\n    model = build_refiner(ns["V8"], pretrained=False)\n    model.load_state_dict(saved["state_dict"], strict=True)\n    return model.to(ns["REFINER_DEVICE"]).eval()\n\n\ndef train_detector(ns, plan, shared_plan):\n    """YOLO training as in V7, plus an explicit nominal batch size (nbs) and warmup length."""\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    directory.mkdir(parents=True, exist_ok=True)\n    identity = dict(plan_sha256=cv_digest(shared_plan), fold=plan["fold"])\n    require_identity(directory / "fold_identity.json", identity)\n    checkpoint, provenance = directory / "yolo_best.pt", directory / "yolo_provenance.json"\n    if provenance.exists():\n        saved = json.loads(provenance.read_text())\n        if saved["identity"] != identity or saved["sha256"] != file_hash(checkpoint):\n            raise ValueError("Saved detector checkpoint provenance mismatch.")\n        print(f"Fold {plan[\'fold\']}: reusing verified detector", flush=True)\n        return YOLO(str(checkpoint))\n    if checkpoint.exists():\n        raise ValueError("Checkpoint publication was interrupted; preserve it and inspect provenance.")\n    if not torch.cuda.is_available() and not ns["SMOKE_TEST"]:\n        raise RuntimeError("Enable a GPU before full training.")\n    settings = ns["V8"]\n    data = prepare_dataset(ns, plan, directory)\n    run = directory / "detector_runs" / "yolo"\n    last, best = run / "weights" / "last.pt", run / "weights" / "best.pt"\n    completed = False\n    if last.exists():\n        # Only read checkpoints generated by this identity-checked local run.\n        saved = torch.load(last, map_location="cpu", weights_only=False)\n        completed = saved["epoch"] < 0 or saved["epoch"] + 1 >= settings["epochs"]\n        del saved\n    started = time.monotonic()\n    if not completed:\n        if last.exists():\n            print(f"Fold {plan[\'fold\']}: resuming last completed YOLO epoch", flush=True)\n            model = YOLO(str(last))\n            # last.pt and best.pt are rewritten every epoch; per-epoch copies (save_period) fill the disk.\n            model.train(resume=True, device=ns["YOLO_TRAIN_DEVICE"], workers=settings["workers"],\n                        save_period=-1)\n        else:\n            print(f"Fold {plan[\'fold\']}: starting YOLO (no saved epoch)", flush=True)\n            model = YOLO(settings["weights"])\n            model.train(\n                data=str(data), epochs=settings["epochs"], imgsz=settings["image_size"],\n                batch=settings["batch_size"], nbs=settings["nbs"],\n                warmup_epochs=settings["warmup_epochs"], device=ns["YOLO_TRAIN_DEVICE"],\n                workers=settings["workers"], patience=settings["patience"], time=None,\n                optimizer="AdamW", lr0=settings["learning_rate"], momentum=0.9,\n                weight_decay=0.0005, warmup_bias_lr=0.0, seed=ns["CFG"]["seed"],\n                single_cls=True, mask_ratio=2, overlap_mask=False,\n                **yolo_augmentation(settings),\n                cache=False, save=True, save_period=-1, plots=False,\n                project=str(run.parent), name=run.name, exist_ok=True,\n            )\n        del model\n        gc.collect()\n    if not best.is_file():\n        raise FileNotFoundError(f"No best detector checkpoint: {best}")\n    temporary = checkpoint.with_suffix(".tmp.pt")\n    shutil.copyfile(best, temporary)\n    temporary.replace(checkpoint)\n    atomic_json(provenance, dict(\n        identity=identity, sha256=file_hash(checkpoint), settings=settings,\n        latest_invocation_seconds=time.monotonic() - started,\n        note="Invocation time excludes earlier interrupted attempts; see YOLO results.csv.",\n    ))\n    return YOLO(str(checkpoint))\n\n\ndef refiner_entries(ns, records):\n    return [(ns["train_lookup"][record["file_name"]], ns["ann_rle"](annotation))\n            for record in records for annotation in ns["annotations_by_id"][record["id"]]]\n\n\ndef train_refiner(ns, plan, shared_plan):\n    """Train the crop refiner on ground-truth instance crops; resume only at fold granularity."""\n    settings, device = ns["V8"], ns["REFINER_DEVICE"]\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    identity = dict(plan_sha256=cv_digest(shared_plan), fold=plan["fold"])\n    checkpoint, provenance = directory / "refiner_best.pt", directory / "refiner_provenance.json"\n    working = directory / "refiner_working_best.pt"\n    if provenance.exists():\n        saved = json.loads(provenance.read_text())\n        if saved["identity"] != identity or saved["sha256"] != file_hash(checkpoint):\n            raise ValueError("Saved refiner checkpoint provenance mismatch.")\n        print(f"Fold {plan[\'fold\']}: reusing verified refiner", flush=True)\n        return load_refiner(ns, checkpoint)\n    for stale in (working, checkpoint):\n        if stale.exists():\n            print(f"Fold {plan[\'fold\']}: discarding unpublished refiner {stale.name}", flush=True)\n            stale.unlink()\n\n    torch.manual_seed(ns["CFG"]["seed"] + plan["fold"])\n    records = fold_records(plan, ns["records_by_name"])\n    crops = CachedFeatureCrops if images_are_rgb(ns) else CachedInstanceCrops\n    datasets = {\n        phase: crops(\n            refiner_entries(ns, records[phase]), ns["CROP_CACHE_DIR"],\n            size=settings["crop_size"], context=settings["crop_context"],\n            minimum=settings["minimum_crop"], augment=phase == "train")\n        for phase in ["train", "tune"]\n    }\n    if crops is CachedFeatureCrops:\n        for dataset in datasets.values():\n            dataset.prewarm(settings["preprocess_workers"])\n    batch = settings["refiner_batch"]\n    loaders = {\n        phase: DataLoader(\n            dataset, batch_size=batch, shuffle=phase == "train",\n            # A one-sample tail (or a one-sample GPU chunk) breaks train-mode BatchNorm.\n            drop_last=phase == "train" and 0 < len(dataset) % batch < 4,\n            **loader_options(settings["refiner_workers"], device.type == "cuda"))\n        for phase, dataset in datasets.items()\n    }\n    if any(len(loader) == 0 for loader in loaders.values()):\n        raise ValueError("Empty crop dataset.")\n    print(f"Fold {plan[\'fold\']}: refiner crops "\n          f"train={len(datasets[\'train\'])} tune={len(datasets[\'tune\'])}", flush=True)\n    gpus = settings["refiner_gpus"] if device.type == "cuda" else []\n    model = parallel_refiner(build_refiner(settings, pretrained=True).to(device), gpus)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["refiner_lr"], weight_decay=1e-4)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=settings["refiner_epochs"])\n    amp = device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    best, significant_best, stale, history, status = float("inf"), float("inf"), 0, [], "max_epochs"\n    started = time.monotonic()\n    for epoch in range(settings["refiner_epochs"]):\n        totals = {}\n        for phase, loader in loaders.items():\n            model.train(phase == "train")\n            total, count = 0.0, 0\n            # Validation draws restart from the same seed every epoch, so its loss is comparable.\n            generator = torch.Generator().manual_seed(1234) if phase == "tune" else None\n            with torch.set_grad_enabled(phase == "train"):\n                for x, y in loader:\n                    x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n                    if phase == "train" and settings.get("refiner_jitter", False):\n                        x, y = jitter_batch(x, y)\n                    elif phase == "train" and settings.get("refiner_rotate", False):\n                        x, y = rotate_batch(x, y)\n                    if settings.get("refiner_mask_channel", False):\n                        coarse = simulate_coarse(y, generator)\n                        x = torch.cat([x, coarse], dim=1)\n                    optimizer.zero_grad(set_to_none=True)\n                    with torch.autocast(device_type=device.type, enabled=amp):\n                        logits = model(x)\n                    loss = segmentation_loss(logits.float(), y.float())\n                    if not torch.isfinite(loss):\n                        raise FloatingPointError("Nonfinite crop loss.")\n                    if phase == "train":\n                        scaler.scale(loss).backward()\n                        scaler.step(optimizer)\n                        scaler.update()\n                    total += loss.item() * len(x)\n                    count += len(x)\n            totals[phase] = total / count\n        history.append(dict(epoch=epoch + 1, learning_rate=optimizer.param_groups[0]["lr"], **totals))\n        print(f"Fold {plan[\'fold\']} REFINER:", history[-1], flush=True)\n        if totals["tune"] < best:\n            best = totals["tune"]\n            temporary = working.with_suffix(".tmp.pt")\n            torch.save(dict(state_dict={k: v.detach().cpu()\n                                        for k, v in base_refiner(model).state_dict().items()},\n                            epoch=epoch + 1), temporary)\n            temporary.replace(working)\n        if totals["tune"] < significant_best - settings["refiner_min_delta"]:\n            significant_best, stale = totals["tune"], 0\n        else:\n            stale += 1\n        scheduler.step()\n        if stale >= settings["refiner_patience"]:\n            status = "early_stopped"\n            break\n    del optimizer, scheduler, scaler, model, loaders\n    gc.collect()\n    if not working.is_file():\n        raise FileNotFoundError("Refiner produced no completed checkpoint.")\n    temporary = checkpoint.with_suffix(".tmp.pt")\n    shutil.copyfile(working, temporary)\n    temporary.replace(checkpoint)\n    atomic_json(directory / "refiner_history.json", history)\n    atomic_json(provenance, dict(\n        identity=identity, sha256=file_hash(checkpoint), settings=settings, status=status,\n        completed_epochs=len(history), seconds=time.monotonic() - started,\n    ))\n    return load_refiner(ns, checkpoint)\n\n\ndef pass_mask(proposal, threshold, shape):\n    """Full-frame mask of the first refiner pass (the detector\'s mask if it comes out empty)."""\n    x0, y0, x1, y1 = proposal["bounds"]\n    probability = np.asarray(Image.fromarray(proposal["probability"]).resize(\n        (x1 - x0, y1 - y0), Image.Resampling.BILINEAR))\n    keep = probability >= threshold\n    if not keep.any():\n        return mask_utils.decode(proposal["coarse"])\n    full = np.zeros(shape, dtype=np.uint8)\n    full[y0:y1, x0:x1] = keep\n    return full\n\n\n@torch.inference_mode()\ndef zoom_pass(image, proposals, refiner, device, shape, settings):\n    """Zoom in on each refined filament and refine again with the first result as channel 4."""\n    size, minimum = settings["crop_size"], settings["minimum_crop"]\n    for threshold in settings["refiner_thresholds"]:\n        firsts = [pass_mask(proposal, threshold, shape) for proposal in proposals]\n        windows = [crop_bounds(first, settings["refiner_zoom_context"], minimum) for first in firsts]\n        for start in range(0, len(proposals), settings["refiner_inference_batch"]):\n            chunk = range(start, min(start + settings["refiner_inference_batch"], len(proposals)))\n            batch = np.stack([np.concatenate([\n                crop_tensor(image, windows[i], size),\n                window_mask(firsts[i], windows[i], size)[None]], axis=0) for i in chunk])\n            probabilities = refiner(torch.from_numpy(batch).to(device)).sigmoid()[:, 0].cpu().numpy()\n            if not np.isfinite(probabilities).all():\n                raise FloatingPointError("Nonfinite zoom probability.")\n            for i, probability in zip(chunk, probabilities):\n                proposals[i].setdefault("zoom", {})[threshold] = dict(\n                    bounds=windows[i], probability=probability)\n    return proposals\n\n\ndef _yolo_source(image):\n    array = np.asarray(image)\n    if array.ndim == 2:\n        array = np.stack([array] * 3, axis=-1)\n    return np.ascontiguousarray(array[..., ::-1])  # OpenCV order, as a file read would give\n\n\n@torch.inference_mode()\ndef detect_proposals(ns, source, detector, refiner, confidence, refine):\n    """YOLO instances for one image (a path or a PIL image); optionally add refiner probabilities."""\n    settings, shape = ns["V8"], tuple(ns["FULL_SHAPE"])\n    if isinstance(source, (str, Path)):\n        with Image.open(source) as opened:\n            opened.load()\n            image = opened.copy()\n        yolo_input = str(source)\n    else:\n        image, yolo_input = source, _yolo_source(source)\n    result = detector.predict(\n        source=yolo_input, imgsz=settings["image_size"], conf=confidence,\n        device=ns["YOLO_INFERENCE_DEVICE"], agnostic_nms=True, retina_masks=True, verbose=False,\n    )[0]\n    if tuple(result.orig_shape) != shape:\n        raise ValueError("Detector result dimensions differ from the image.")\n    if result.masks is None or len(result.masks) == 0:\n        return []\n    masks = (result.masks.data > 0.5).to(torch.uint8).cpu().numpy()\n    if masks.ndim != 3 or tuple(masks.shape[1:]) != shape:\n        raise ValueError("Detector masks must retain original image dimensions.")\n    if refine:\n        proposals = refine_image(\n            image, masks, refiner, ns["REFINER_DEVICE"], shape,\n            batch_size=settings["refiner_inference_batch"], crop_size=settings["crop_size"],\n            context=settings["crop_context"], minimum=settings["minimum_crop"],\n            mask_channel=settings.get("refiner_mask_channel", False),\n        )\n        if settings.get("refiner_zoom", False):\n            zoom_pass(image, proposals, refiner, ns["REFINER_DEVICE"], shape, settings)\n        return proposals\n    return [dict(coarse=mask_utils.encode(np.asfortranarray(mask)), bounds=None, probability=None)\n            for mask in masks if mask.any()]\n\n\ndef largest_component(mask):\n    """Keep the biggest 8-connected piece: every ground-truth filament is one polygon."""\n    labels, count = ndi.label(mask, structure=np.ones((3, 3), dtype=np.uint8))\n    if count <= 1:\n        return mask\n    return labels == (np.argmax(np.bincount(labels.ravel())[1:]) + 1)\n\n\ndef build_instances(proposals, parameters, shape):\n    """Turn proposals into disjoint instances under one yolo/refined/gated parameter set."""\n    mode = parameters["mode"]\n    if mode not in MODES:\n        raise ValueError(f"Unknown mode: {mode}")\n    instances, fallbacks = [], 0\n    for proposal in proposals:\n        rle = proposal["coarse"]\n        if mode == "zoom":\n            x0, y0, x1, y1 = proposal["zoom"][parameters["refiner_threshold"]]["bounds"]\n            probability = np.asarray(Image.fromarray(\n                proposal["zoom"][parameters["refiner_threshold"]]["probability"]).resize(\n                    (x1 - x0, y1 - y0), Image.Resampling.BILINEAR))\n            keep = probability >= parameters["refiner_threshold"]\n            if parameters.get("single_piece", False) and keep.any():\n                keep = largest_component(keep)\n            if keep.any():\n                full = np.zeros(shape, dtype=np.uint8)\n                full[y0:y1, x0:x1] = keep\n                rle = mask_utils.encode(np.asfortranarray(full))\n            else:  # fall back to the first pass\n                fallbacks += 1\n                rle = mask_utils.encode(np.asfortranarray(\n                    pass_mask(proposal, parameters["refiner_threshold"], shape)))\n        elif mode != "yolo":\n            x0, y0, x1, y1 = proposal["bounds"]\n            probability = np.asarray(Image.fromarray(proposal["probability"]).resize(\n                (x1 - x0, y1 - y0), Image.Resampling.BILINEAR))\n            keep = probability >= parameters["refiner_threshold"]\n            if mode == "gated":\n                window = mask_utils.decode(rle)[y0:y1, x0:x1].astype(bool)\n                keep &= ndi.binary_dilation(window, iterations=parameters["gate_pixels"])\n            if parameters.get("single_piece", False) and keep.any():\n                keep = largest_component(keep)\n            if keep.any():\n                full = np.zeros(shape, dtype=np.uint8)\n                full[y0:y1, x0:x1] = keep\n                rle = mask_utils.encode(np.asfortranarray(full))\n            else:\n                fallbacks += 1\n        if mask_utils.area(rle) >= parameters["min_area"]:\n            instances.append(rle)\n    instances, _ = disjoint_instances(instances, shape, parameters["min_area"])\n    return instances, fallbacks\n\n\ndef candidate_parameters(settings, confidence):\n    for area in settings["min_areas"]:\n        base = dict(confidence=confidence, min_area=area)\n        if settings.get("single_piece", False):\n            base["single_piece"] = True\n        yield dict(mode="yolo", refiner_threshold=None, gate_pixels=None, **base)\n        for threshold in settings["refiner_thresholds"]:\n            yield dict(mode="refined", refiner_threshold=threshold, gate_pixels=None, **base)\n            for gate in settings["gate_pixels"]:\n                yield dict(mode="gated", refiner_threshold=threshold, gate_pixels=gate, **base)\n            if settings.get("refiner_zoom", False):\n                yield dict(mode="zoom", refiner_threshold=threshold, gate_pixels=None, **base)\n\n\n# Test-time voting: the 8 rotations/flips of a square frame are lossless, and V11 trains on all\n# of them. Each view is processed by the full pipeline, its masks are mapped back, and the\n# views vote on which instances exist and which pixels belong to them.\nVIEWS = [(k, flip) for flip in (False, True) for k in range(4)]\n\n\ndef apply_view(array, k, flip):\n    view = np.rot90(array, k, axes=(0, 1))\n    if flip:\n        view = view[:, ::-1]\n    return np.ascontiguousarray(view)\n\n\ndef invert_mask(mask, k, flip):\n    if flip:\n        mask = mask[:, ::-1]\n    return np.ascontiguousarray(np.rot90(mask, -k, axes=(0, 1)))\n\n\ndef tta_view_lists(ns, path, detector, refiner, parameters, count):\n    """Instances of one image under `count` views, each mapped back to the original frame."""\n    shape = tuple(ns["FULL_SHAPE"])\n    with Image.open(path) as opened:\n        opened.load()\n        image = opened.copy()\n    array = np.asarray(image)\n    lists = []\n    for k, flip in VIEWS[:count]:\n        view = image if (k, flip) == (0, False) else Image.fromarray(apply_view(array, k, flip))\n        proposals = detect_proposals(ns, view, detector, refiner, parameters["confidence"],\n                                     parameters["mode"] != "yolo")\n        instances, _ = build_instances(proposals, parameters, shape)\n        if (k, flip) != (0, False):\n            instances = [mask_utils.encode(np.asfortranarray(\n                invert_mask(mask_utils.decode(rle), k, flip))) for rle in instances]\n        lists.append(instances)\n    return lists\n\n\ndef cluster_views(view_lists, match_iou=0.5):\n    """Group instances across views: one-to-one IoU matching, at most one member per view."""\n    clusters = []\n    for view_index, instances in enumerate(view_lists):\n        if not instances:\n            continue\n        matched = set()\n        if clusters:\n            overlaps = mask_utils.iou([c["rep"] for c in clusters], instances, [0] * len(instances))\n            eligible = overlaps >= match_iou\n            weights = eligible * (min(overlaps.shape) + 1 + overlaps)\n            for row, column in zip(*linear_sum_assignment(-weights)):\n                if eligible[row, column]:\n                    clusters[row]["members"].append(instances[column])\n                    matched.add(column)\n        clusters.extend(dict(rep=rle, members=[rle])\n                        for column, rle in enumerate(instances) if column not in matched)\n    return clusters\n\n\ndef cluster_votes(clusters, shape, minimum_support):\n    """Per-pixel vote counts (how many views include each pixel) for supported clusters."""\n    votes = {}\n    for index, cluster in enumerate(clusters):\n        if len(cluster["members"]) >= minimum_support:\n            total = np.zeros(shape, dtype=np.uint8)\n            for rle in cluster["members"]:\n                total += mask_utils.decode(rle)\n            votes[index] = total\n    return votes\n\n\ndef fuse_clusters(clusters, votes, instance_votes, pixel_votes, min_area, shape, single_piece=False):\n    """Keep instances seen in >= instance_votes views; keep pixels in >= pixel_votes views."""\n    kept = []\n    for index, total in votes.items():\n        support = len(clusters[index]["members"])\n        if support < instance_votes:\n            continue\n        mask = total >= pixel_votes\n        if single_piece and mask.any():\n            mask = largest_component(mask)\n        area = int(mask.sum())\n        if area >= min_area:\n            kept.append((support, area, mask))\n    kept.sort(key=lambda item: (-item[0], -item[1]))\n    rles = [mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))) for _, _, mask in kept]\n    instances, _ = disjoint_instances(rles, shape, min_area)\n    return instances\n\n\ndef tta_grid(settings):\n    return [(v, p) for v in settings["tta_instance_votes"] for p in settings["tta_pixel_votes"]]\n\n\ndef tta_parameters(parameters, choice):\n    return dict(parameters, tta=dict(views=choice["views"], instance_votes=choice["instance_votes"],\n                                     pixel_votes=choice["pixel_votes"]))\n\n\ndef select_tta(ns, detector, refiner, plan, identity, calibration):\n    """Choose voting thresholds on the tuning photos; vote only if it beats the single view."""\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    path = directory / "tta.json"\n    if path.exists():\n        saved = json.loads(path.read_text())\n        if saved["identity"] != identity:\n            raise ValueError("Voting selection provenance mismatch.")\n        return saved\n    settings, shape = ns["V8"], tuple(ns["FULL_SHAPE"])\n    count = settings["tta_views"]\n    names = calibration["tuning_names"]\n    parameters = calibration["selected"]["parameters"]\n    cache_path = directory / "tta_tuning_views.json"\n    expected = dict(identity=identity, parameters=parameters, views=count, names=names)\n    cache = dict(identity=expected, views={})\n    if cache_path.exists():\n        cache = json.loads(cache_path.read_text())\n        if cache["identity"] != expected:\n            raise ValueError(f"Stale voting cache: {cache_path}")\n    started = time.monotonic()\n    for index, name in enumerate(names, 1):\n        stem = Path(name).stem\n        if stem in cache["views"]:\n            continue\n        lists = tta_view_lists(ns, ns["train_lookup"][name], detector, refiner, parameters, count)\n        cache["views"][stem] = [encode_masks(instances) for instances in lists]\n        if index % SAVE_EVERY == 0 or index == len(names):\n            atomic_json(cache_path, cache)\n            print(f"Voting views: {len(cache[\'views\'])}/{len(names)} tuning photos "\n                  f"({time.monotonic() - started:.0f}s)", flush=True)\n    grid = tta_grid(settings)\n    predictions = {combo: {} for combo in grid}\n    lowest = min(v for v, _ in grid)\n    for name in names:\n        stem = Path(name).stem\n        lists = [decode_masks(items) for items in cache["views"][stem]]\n        clusters = cluster_views(lists)\n        votes = cluster_votes(clusters, shape, lowest)\n        for combo in grid:\n            instances = fuse_clusters(clusters, votes, combo[0], combo[1], parameters["min_area"], shape,\n                                      parameters.get("single_piece", False))\n            predictions[combo][stem] = encode_masks(instances)\n    trials = []\n    for combo in grid:\n        means, _ = score_predictions(ns, names, predictions[combo])\n        trials.append(dict(instance_votes=combo[0], pixel_votes=combo[1], scores=means))\n    best = max(trials, key=lambda trial: trial["scores"]["pq"])\n    single = calibration["selected"]["scores"]["pq"]\n    choice = dict(identity=identity, views=count, instance_votes=best["instance_votes"],\n                  pixel_votes=best["pixel_votes"], enabled=best["scores"]["pq"] > single,\n                  tuning_pq_voting=best["scores"]["pq"], tuning_pq_single=single, trials=trials)\n    atomic_json(path, choice)\n    print("VOTING (tuning only): best", (best["instance_votes"], best["pixel_votes"]),\n          "PQ %.4f vs single view %.4f -> %s" % (best["scores"]["pq"], single,\n                                                  "using voting" if choice["enabled"] else "single view"),\n          flush=True)\n    return choice\n\n\ndef tta_predictor(ns, detector, refiner, parameters, choice, counters):\n    """Per-image voting predictor; a failure on one image falls back to its single view."""\n    shape = tuple(ns["FULL_SHAPE"])\n\n    def predict(path):\n        try:\n            lists = tta_view_lists(ns, path, detector, refiner, parameters, choice["views"])\n            clusters = cluster_views(lists)\n            votes = cluster_votes(clusters, shape, choice["instance_votes"])\n            return fuse_clusters(clusters, votes, choice["instance_votes"], choice["pixel_votes"],\n                                 parameters["min_area"], shape, parameters.get("single_piece", False))\n        except Exception as error:  # keep the run alive; report it\n            counters["fallbacks"] += 1\n            print(f"WARNING: voting failed for {Path(path).name} ({error}); using the single view",\n                  flush=True)\n            proposals = detect_proposals(ns, path, detector, refiner, parameters["confidence"],\n                                         parameters["mode"] != "yolo")\n            return build_instances(proposals, parameters, shape)[0]\n\n    return predict\n\n\ndef _cache_identity(identity, parameters, paths, shape):\n    return dict(model=identity, parameters=parameters,\n                images_sha256=cv_digest(image_manifest(paths)), shape=list(shape))\n\n\ndef _read_cache(cache_path, expected_identity, expected, shape):\n    cache_path = Path(cache_path)\n    if not cache_path.exists():\n        return dict(identity=expected_identity, predictions={})\n    cache = json.loads(cache_path.read_text())\n    if cache["identity"] != expected_identity:\n        raise ValueError(f"Stale prediction cache: {cache_path}")\n    if not set(cache["predictions"]) <= expected:\n        raise ValueError("Prediction cache contains unknown image IDs.")\n    for items in cache["predictions"].values():\n        validate_disjoint(decode_masks(items), shape)\n    return cache\n\n\ndef predict_cached(ns, detector, refiner, paths, parameters, cache_path, identity, predictor=None):\n    """Save every few images; an interrupted pass restarts only the missing ones."""\n    paths, shape = list(paths), tuple(ns["FULL_SHAPE"])\n    expected = {p.stem for p in paths}\n    cache = _read_cache(cache_path, _cache_identity(identity, parameters, paths, shape), expected, shape)\n    predictions = cache["predictions"]\n    missing = [p for p in paths if p.stem not in predictions]\n    print(f"{Path(cache_path).name}: {len(predictions)}/{len(paths)} saved", flush=True)\n    for index, path in enumerate(missing, 1):\n        if predictor is None:\n            proposals = detect_proposals(ns, path, detector, refiner,\n                                         parameters["confidence"], parameters["mode"] != "yolo")\n            instances, _ = build_instances(proposals, parameters, shape)\n        else:\n            instances = predictor(path)\n        validate_disjoint(instances, shape)\n        predictions[path.stem] = encode_masks(instances)\n        if index % SAVE_EVERY == 0 or index == len(missing):\n            atomic_json(cache_path, cache)\n            print(f"{Path(cache_path).name}: {len(predictions)}/{len(paths)} saved", flush=True)\n    if set(predictions) != expected:\n        raise ValueError("Incomplete inference coverage.")\n    return predictions\n\n\ndef load_cached_predictions(ns, paths, parameters, cache_path, identity):\n    paths, shape = list(paths), tuple(ns["FULL_SHAPE"])\n    expected = {p.stem for p in paths}\n    if not Path(cache_path).exists():\n        raise FileNotFoundError(f"Missing test predictions: {cache_path}")\n    cache = _read_cache(cache_path, _cache_identity(identity, parameters, paths, shape), expected, shape)\n    if set(cache["predictions"]) != expected:\n        raise ValueError(f"Incomplete test predictions: {cache_path}")\n    return cache["predictions"]\n\n\ndef fold_model_identity(ns, plan, shared_plan):\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    return dict(plan_sha256=cv_digest(shared_plan), fold=plan["fold"],\n                checkpoint_sha256=file_hash(directory / "yolo_best.pt"),\n                refiner_sha256=file_hash(directory / "refiner_best.pt"))\n\n\ndef checkpoint_hashes(identity):\n    return {"yolo_best.pt": identity["checkpoint_sha256"],\n            "refiner_best.pt": identity["refiner_sha256"]}\n\n\ndef calibrate(ns, detector, refiner, plan, identity):\n    """Pick confidence/mode on the inner tuning days only; proposals are computed once per confidence."""\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    path = directory / "calibration.json"\n    if path.exists():\n        saved = json.loads(path.read_text())\n        if saved["identity"] != identity:\n            raise ValueError("Calibration provenance mismatch.")\n        return saved\n    settings, shape = ns["V8"], tuple(ns["FULL_SHAPE"])\n    names = sorted(plan["splits"]["tune"])\n    random.Random(ns["CFG"]["seed"] + plan["fold"]).shuffle(names)\n    names = names[:settings["tuning_images"]]\n    if not names:\n        raise ValueError("No tuning images.")\n    trials = []\n    for confidence in settings["confidences"]:\n        started = time.monotonic()\n        cache = {name: detect_proposals(ns, ns["train_lookup"][name], detector, refiner, confidence, True)\n                 for name in names}\n        for parameters in candidate_parameters(settings, confidence):\n            predictions, fallbacks = {}, 0\n            for name in names:\n                instances, used = build_instances(cache[name], parameters, shape)\n                predictions[Path(name).stem] = encode_masks(instances)\n                fallbacks += used\n            means, _ = score_predictions(ns, names, predictions)\n            trials.append(dict(parameters=parameters, scores=means, refiner_fallbacks=fallbacks))\n        del cache\n        print(f"Fold {plan[\'fold\']} tuning conf={confidence}: "\n              f"{time.monotonic() - started:.0f}s", flush=True)\n    best_by_mode = {\n        mode: max((t for t in trials if t["parameters"]["mode"] == mode),\n                  key=lambda t: t["scores"]["pq"])\n        for mode in MODES if any(t["parameters"]["mode"] == mode for t in trials)\n    }\n    selected = max(trials, key=lambda t: t["scores"]["pq"])\n    result = dict(identity=identity, selected=selected, best_by_mode=best_by_mode,\n                  trials=trials, tuning_names=names)\n    atomic_json(path, result)\n    print("TUNING ONLY, not independent performance:",\n          {mode: (t["parameters"], round(t["scores"]["pq"], 4)) for mode, t in best_by_mode.items()},\n          "\\nSELECTED:", selected["parameters"], flush=True)\n    return result\n\n\ndef merge_holdout_into_training(plan):\n    """Train on the holdout days too; only the tuning days stay out of training."""\n    splits = plan["splits"]\n    if set(splits["train"]) & set(splits["holdout"]):\n        raise ValueError("Train and holdout days overlap.")\n    return dict(plan, splits=dict(\n        train=sorted(splits["train"] + splits["holdout"]), tune=list(splits["tune"]), holdout=[]))\n\n\ndef validate_result(result, plan, plan_hash):\n    """validate_fold_result needs outer rows; without a holdout check the rest."""\n    if plan["splits"]["holdout"]:\n        return validate_fold_result(result, plan, plan_hash)\n    if (result["plan_sha256"] != plan_hash or result["fold"] != plan["fold"]\n            or result["status"] != "complete" or result["outer_rows"]\n            or not result["checkpoint_hashes"] or not result["selected_parameters"]):\n        raise ValueError("Fold result provenance/status mismatch.")\n\n\ndef run_fold(ns, plan, shared_plan):\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    path = directory / "fold_result.json"\n    plan_hash = cv_digest(shared_plan)\n    if path.exists():\n        result = json.loads(path.read_text())\n        validate_result(result, plan, plan_hash)\n        if result["checkpoint_hashes"] != checkpoint_hashes(fold_model_identity(ns, plan, shared_plan)):\n            raise ValueError("Completed fold checkpoint changed.")\n        print(f"Fold {plan[\'fold\']}: reusing completed run", flush=True)\n        return result\n    started = time.monotonic()\n    detector, refiner = train_detector(ns, plan, shared_plan), None\n    try:\n        refiner = train_refiner(ns, plan, shared_plan)\n        identity = fold_model_identity(ns, plan, shared_plan)\n        calibration = calibrate(ns, detector, refiner, plan, identity)\n        parameters = calibration["selected"]["parameters"]\n        names = plan["splits"]["holdout"]\n        means, rows, yolo_means = None, [], None\n        yolo_parameters = calibration["best_by_mode"]["yolo"]["parameters"]\n        if names:\n            outer_paths = [ns["train_lookup"][name] for name in names]\n            predictions = predict_cached(ns, detector, refiner, outer_paths, parameters,\n                                         directory / "outer_predictions.json", identity)\n            means, rows = score_predictions(ns, names, predictions)\n            # Matched diagnostic: same detector and split, crop refinement switched off.\n            yolo_means = means\n            if parameters["mode"] != "yolo":\n                yolo_predictions = predict_cached(\n                    ns, detector, refiner, outer_paths, yolo_parameters,\n                    directory / "outer_predictions_yolo_only.json", identity)\n                yolo_means, _ = score_predictions(ns, names, yolo_predictions)\n        predict_cached(ns, detector, refiner, ns["test_paths"], parameters,\n                       directory / "test_predictions.json", identity)\n        choice = dict(enabled=False)\n        counters = dict(fallbacks=0)\n        if ns["V8"].get("tta_views", 1) > 1:\n            choice = select_tta(ns, detector, refiner, plan, identity, calibration)\n            if choice["enabled"]:\n                predict_cached(ns, detector, refiner, ns["test_paths"], tta_parameters(parameters, choice),\n                               directory / "test_predictions_tta.json", identity,\n                               predictor=tta_predictor(ns, detector, refiner, parameters, choice, counters))\n        result = dict(\n            fold=plan["fold"], plan_sha256=plan_hash, status="complete",\n            smoke_test=ns["SMOKE_TEST"], preprocessing=ns["PREPROCESSING"],\n            checkpoint_hashes=checkpoint_hashes(identity),\n            selected_parameters=parameters, inner_selected_scores=calibration["selected"]["scores"],\n            outer_mean=means, outer_rows=rows,\n            outer_yolo_only_parameters=yolo_parameters, outer_yolo_only_mean=yolo_means,\n            tuning_pq_by_mode={mode: trial["scores"]["pq"]\n                               for mode, trial in calibration["best_by_mode"].items()},\n            training_photos=len(plan["splits"]["train"]),\n            tta={k: v for k, v in choice.items() if k != "trials" and k != "identity"},\n            tta_fallbacks=counters["fallbacks"],\n            fold_seconds_latest_invocation=time.monotonic() - started,\n        )\n        validate_result(result, plan, plan_hash)\n        atomic_json(path, result)\n        return result\n    finally:\n        del detector, refiner\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n\ndef crop_report(results):\n    """Per-fold view of what the crop stage did; holdout PQs exist only with a holdout."""\n    rows = []\n    for result in sorted(results, key=lambda item: item["fold"]):\n        selected = result["selected_parameters"]\n        row = dict(\n            fold=result["fold"], mode=selected["mode"], confidence=selected["confidence"],\n            refiner_threshold=selected["refiner_threshold"], gate_pixels=selected["gate_pixels"],\n            tuning_pq=round(result["inner_selected_scores"]["pq"], 4),\n            tuning_pq_by_mode={k: round(v, 4) for k, v in result["tuning_pq_by_mode"].items()},\n            training_photos=result["training_photos"],\n            voting=result.get("tta", {}), voting_fallbacks=result.get("tta_fallbacks", 0),\n        )\n        if result["outer_mean"] is not None:\n            row.update(\n                outer_pq_selected=round(result["outer_mean"]["pq"], 4),\n                outer_pq_yolo_only=round(result["outer_yolo_only_mean"]["pq"], 4),\n                outer_instances_per_image=round(result["outer_mean"]["instances"], 2))\n        rows.append(row)\n    return rows\n\n\ndef export_single(ns, plan, shared_plan):\n    """Write the one trained model\'s cached test predictions as a validated CSV."""\n    if ns["SMOKE_TEST"] or shared_plan["smoke_test"]:\n        raise ValueError("Smoke experiments cannot export competition submissions.")\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    result = json.loads((directory / "fold_result.json").read_text())\n    validate_result(result, plan, cv_digest(shared_plan))\n    identity = fold_model_identity(ns, plan, shared_plan)\n    if result["checkpoint_hashes"] != checkpoint_hashes(identity):\n        raise ValueError("Export checkpoints differ from the evaluated models.")\n    calibration = json.loads((directory / "calibration.json").read_text())\n    if (calibration["identity"] != identity\n            or calibration["selected"]["parameters"] != result["selected_parameters"]):\n        raise ValueError("Export calibration differs from evaluated settings.")\n    root, shape, paths = ns["CV_ROOT"], tuple(ns["FULL_SHAPE"]), ns["test_paths"]\n    names = {path.stem for path in paths}\n    if result.get("tta", {}).get("enabled"):\n        predictions = load_cached_predictions(\n            ns, paths, tta_parameters(result["selected_parameters"], result["tta"]),\n            directory / "test_predictions_tta.json", identity)\n    else:\n        predictions = load_cached_predictions(\n            ns, paths, result["selected_parameters"], directory / "test_predictions.json", identity)\n    rows = []\n    for path in paths:\n        instances = decode_masks(predictions[path.stem])\n        validate_disjoint(instances, shape)\n        rows.extend(dict(filament_id=f"{path.stem}_{i}", segmentation_rle=rle["counts"].decode("ascii"))\n                    for i, rle in enumerate(instances, 1))\n    validate_submission_rows(rows, names, shape)\n    version = ns.get("VERSION", "V8")\n    destination = root / f"submission_{version}_{ns[\'PREPROCESSING\']}.csv"\n    temporary = destination.with_suffix(".tmp.csv")\n    with temporary.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=SUBMISSION_FIELDS)\n        writer.writeheader()\n        writer.writerows(rows)\n    with temporary.open(newline="") as stream:\n        reader = csv.DictReader(stream)\n        if reader.fieldnames != SUBMISSION_FIELDS:\n            raise ValueError("Written CSV columns differ.")\n        readback = list(reader)\n    if readback != rows:\n        raise ValueError("Written CSV differs from generated predictions.")\n    audit = validate_submission_rows(readback, names, shape)\n    audit["masks_per_image_with_predictions"] = round(\n        audit["rows"] / audit["images_with_predictions"], 2)\n    digest = file_hash(temporary)\n    temporary.replace(destination)\n    top_level = root.parent / "submission.csv"\n    shutil.copyfile(destination, top_level)\n    atomic_json(root / "submission_audit.json", dict(\n        run_name=root.name, preprocessing=ns["PREPROCESSING"],\n        architecture="single YOLO11s-seg + ResNet34 U-Net 256 px crop refiner",\n        plan_sha256=cv_digest(shared_plan), hashes=result["checkpoint_hashes"],\n        parameters=result["selected_parameters"], voting=result.get("tta"),\n        voting_fallbacks=result.get("tta_fallbacks", 0),\n        test_manifest_sha256=cv_digest(image_manifest(paths)),\n        validation=audit, submission_filename=destination.name, submission_sha256=digest,\n        copied_to=str(top_level),\n    ))\n    if file_hash(destination) != digest or file_hash(top_level) != digest:\n        raise ValueError("Final submission checksum mismatch.")\n    print(f"FINAL {version} CSV VERIFIED:", destination, "\\nCopied to:", top_level, "\\nSHA256:", digest,\n          "\\nVALIDATION:", audit, flush=True)\n    return destination\n\n\ndef verify_ready_to_submit(ns, path, min_masks=2.0, max_masks=10.5, force=False):\n    """Re-check the exported CSV against its audit before anything is uploaded."""\n    path = Path(path)\n    audit = json.loads((ns["CV_ROOT"] / "submission_audit.json").read_text())\n    validation = audit["validation"]\n    if ns["SMOKE_TEST"] or not path.is_file() or path.name != audit["submission_filename"]:\n        raise ValueError("Submission file is missing or is not the audited CSV.")\n    if file_hash(path) != audit["submission_sha256"]:\n        raise ValueError("Submission checksum differs from its audit.")\n    if validation["overlapping_pixels"] != 0:\n        raise ValueError("Submission masks overlap.")\n    if validation["images_with_predictions"] < 0.9 * len(ns["test_paths"]):\n        raise ValueError("Fewer than 90% of test images have predictions.")\n    density = validation["masks_per_image_with_predictions"]\n    if not force and not min_masks <= density <= max_masks:\n        raise ValueError(\n            f"{density} masks per image is outside {min_masks}-{max_masks} (labels average about 7.1; "\n            "V7 scored 0.26 PQ at about 11.6). Set FORCE_SUBMIT = True to upload anyway.")\n    return audit\n')
if str(runtime_dir) not in sys.path:
    sys.path.insert(0, str(runtime_dir))
from grouped_yolo_cv import cv_digest, fold_records, make_day_folds
from v8_runtime import (
    crop_report, export_single, merge_holdout_into_training, prepare_images, run_fold,
    validate_result, verify_ready_to_submit,
)


## Solar-disk normalization
Writes a normalized copy of all 887 frames (about 2-4 minutes on 4 CPUs, skipped when `DISK_NORMALIZE = False`) and points every later stage at them. The fitted center and radius of each frame are saved to `disk_fits.json`.

In [ ]:
raw_train_lookup, raw_test_paths = train_lookup, test_paths
disk_report = None
if DISK_NORMALIZE:
    train_lookup, test_paths, disk_report = prepare_images(globals())
    sample = sorted(train_lookup)[0]
    print("Normalized frame example:", train_lookup[sample])
else:
    print("DISK_NORMALIZE is False: training on the raw frames.")


## One day-grouped split
Image days are isolated to train and tuning (plus an outer holdout when `TRAIN_ON_HOLDOUT = False`). The refiner trains on ground-truth instance crops from the training days and early-stops on the tuning days.

In [ ]:
if FOLDS_TO_RUN != [SPLIT_FOLD] or SPLIT_FOLD not in range(5):
    raise ValueError("V8 trains exactly one split: set SPLIT_FOLD to 0-4.")
cv_plans = make_day_folds(records_by_name, CFG["seed"], inner_fraction=INNER_TUNE_FRACTION)
plan = cv_plans[SPLIT_FOLD]
if TRAIN_ON_HOLDOUT:
    plan = merge_holdout_into_training(plan)
runtime_sha256 = hashlib.sha256("".join(
    (runtime_dir / name).read_text() for name in sorted(p.name for p in runtime_dir.glob("*.py"))
).encode()).hexdigest()
cv_plan = dict(
    name=RUN_NAME, seed=CFG["seed"], annotation_sha256=annotation_sha256,
    settings=V8, inner_tune_fraction=INNER_TUNE_FRACTION, preprocessing=PREPROCESSING,
    train_on_holdout=TRAIN_ON_HOLDOUT, split=plan, smoke_test=SMOKE_TEST,
    architecture="single YOLO11m-seg + ResNet34 U-Net crop refiner; yolo/refined/gated modes",
    runtime_sha256=runtime_sha256,
)
plan_path = CV_ROOT / "cv_plan.json"
if plan_path.exists() and json.loads(plan_path.read_text()) != cv_plan:
    raise ValueError("Existing V8 run plan differs. Choose a new RUN_NAME.")
if not plan_path.exists():
    plan_path.write_text(json.dumps(cv_plan, indent=2, allow_nan=False))
cv_plan_hash = cv_digest(cv_plan)
entries = fold_records(plan, records_by_name)
print("Split", SPLIT_FOLD, {
    phase: dict(physical_images=len(plan["splits"][phase]), annotator_sets=len(selected),
                instances=sum(len(annotations_by_id[r["id"]]) for r in selected))
    for phase, selected in entries.items()
})
print("Plan SHA256:", cv_plan_hash)
with (CV_ROOT / "requirements-actual.txt").open("w") as stream:
    subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                   stdout=stream, check=True)


## Train, calibrate, evaluate, and predict
YOLO (up to 100 epochs), then the refiner, then calibration on the tuning images, then the outer holdout when one exists (selected mode and the matched YOLO-only comparison), then the 180 test images.

In [ ]:
import time

RUN_STARTED = time.monotonic()
cv_result = run_fold(globals(), plan, cv_plan)
print("TUNING (selected mode)", PREPROCESSING, cv_result["inner_selected_scores"], flush=True)
if cv_result["outer_mean"] is not None:
    print("OUTER HOLDOUT", cv_result["outer_mean"], flush=True)
print(f"Elapsed: {(time.monotonic() - RUN_STARTED) / 60:.1f} min", flush=True)


## Review
Tuning scores were used for selection and are optimistic; outer scores (only when a holdout exists) are never used for selection. Run again with `DISK_NORMALIZE` flipped for the matched comparison.

In [ ]:
cv_result = json.loads((CV_ROOT / f"fold_{SPLIT_FOLD}" / "fold_result.json").read_text())
validate_result(cv_result, plan, cv_plan_hash)
print(json.dumps(dict(
    preprocessing=cv_result["preprocessing"], training_photos=cv_result["training_photos"],
    tuning_selected=cv_result["inner_selected_scores"],
    tuning_pq_by_mode=cv_result["tuning_pq_by_mode"],
    holdout_selected_mode=cv_result["outer_mean"],
    holdout_yolo_only=cv_result["outer_yolo_only_mean"],
), indent=2, allow_nan=False))
print("Crop stage (tuning-selected mode; holdout PQ with and without refinement):")
for row in crop_report([cv_result]):
    print(row)
print("Compare tuning_pq (or outer_pq_selected when a holdout exists) between the disk_normalized and raw runs.")


## Prepare the CSV
Validates disjoint masks, CSV readback and image coverage, and copies the file to `/kaggle/working/submission.csv`. Check `masks_per_image_with_predictions`: labeled data average about 7.1 instances per annotator set; V7's 0.26 PQ CSV had about 11.6.

In [ ]:
submission_path = export_single(globals(), plan, cv_plan)


## Check and submit manually
Prints the audit of the written CSV (name, SHA-256, overlaps, image coverage, masks per image) as warnings only. Then submit the file yourself, for example:

```bash
kaggle competitions submit -c filament-segmentation-2026 -f submission_V10_disk_normalized.csv -m "V10: single YOLO + crop refiner, solar-disk normalization"
```

In [ ]:
try:
    audit = verify_ready_to_submit(globals(), submission_path)
    print("CSV checks passed:", audit["validation"])
except Exception as error:  # report only; never stop Run All
    print("WARNING: pre-submit check did not pass:", error)
    print("The CSV was still written; inspect it before submitting.")
print("File:", submission_path, "| also copied to", CV_ROOT.parent / "submission.csv")
print("Submit with: kaggle competitions submit -c", COMPETITION, "-f", submission_path,
      "-m", "V10 single model " + PREPROCESSING)
